# 02_03 - Limpieza base de oferta espacial SER

Este notebook prepara fuentes limpias individuales del bloque espacial SER en la fase de descripción y limpieza de datos. El objetivo no es construir todavía oferta agregada, panel SER, joins finales ni métricas proxy, sino dejar cada fuente en `data/interim/ser/...` con un esquema mínimo, validado y trazable.

**Fuentes tratadas, en orden metodológico:**

1. `ser_zonas`: lookup auxiliar vía/rango/paridad → zona SER.
2. `ser_geoportal_limite_ser`: polígono oficial del área SER.
3. `ser_geoportal_barrios_ser`: polígonos de barrios SER.
4. `ser_geoportal_bandas_aparcamiento`: geometría lineal de bandas y plazas reguladas.
5. `ser_calles_plazas`: capacidad tabular histórica por año, calle/finca/color/plazas.
6. `ser_parquimetros`: infraestructura SER con matrícula, vigencia, calle y coordenadas.

La estructura de cada bloque sigue la misma lógica: primero se explica qué mide la fuente y qué columnas se conservan o descartan; después se ejecutan checks de calidad; a continuación se toma una decisión de limpieza; por último se muestra la tabla limpia resultante cuando aporta información.


## 0. Configuración inicial

Se inicializan rutas, dependencias y constantes. La raíz se detecta automáticamente con `find_repo_root()` a partir de `data_catalog.csv`; cualquier ruta absoluta impresa es solo un diagnóstico local de ejecución, no una dependencia hardcodeada.


In [1]:
from __future__ import annotations

import glob
import json
import re
import tempfile
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
except ImportError as exc:
    raise ImportError(
        "Este notebook requiere geopandas para las capas Geoportal. "
        "Instalar con conda install -c conda-forge geopandas pyogrio shapely pyarrow rtree"
    ) from exc

try:
    from shapely.geometry import Point
except ImportError as exc:
    raise ImportError(
        "Este notebook requiere geopandas para las capas Geoportal. "
        "Instalar con conda install -c conda-forge geopandas pyogrio shapely pyarrow rtree"
    ) from exc

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 90)
pd.set_option("display.max_colwidth", 160)

TARGET_DATASET_IDS = [
    "ser_zonas",
    "ser_geoportal_limite_ser",
    "ser_geoportal_barrios_ser",
    "ser_geoportal_bandas_aparcamiento",
    "ser_calles_plazas",
    "ser_parquimetros",
]

WINDOW_CALLES = {2023, 2024, 2025, 2026}
SER_WINDOW_START = pd.Timestamp("2023-01-01")
SER_WINDOW_END = pd.Timestamp("2026-12-31")
SER_REGULATED_COLORS = ["Azul", "Verde", "Alta Rotación", "Rojo", "Naranja"]
SER_REGULATED_COLORS_NORM = {"azul", "verde", "alta rotacion", "rojo", "naranja"}
SER_COLORS_NORM = SER_REGULATED_COLORS_NORM | {"gris"}


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv. Ejecuta el notebook desde la raiz del repo TFM_parking_madrid."
    )


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"
CHECKLIST_PATH = ROOT / "docs" / "limpieza_SER_checklist.md"
SOURCE_DOCS_ROOT = ROOT / "docs" / "source_docs" / "ser"

print(f"ROOT = {ROOT}")
print(f"CATALOG_PATH exists = {CATALOG_PATH.exists()}")
print(f"CHECKLIST_PATH exists = {CHECKLIST_PATH.exists()}")
print(f"SOURCE_DOCS_ROOT exists = {SOURCE_DOCS_ROOT.exists()}")
print(f"geopandas = {gpd.__version__}")


ROOT = /Users/hugo/TFM_parking_madrid
CATALOG_PATH exists = True
CHECKLIST_PATH exists = True
SOURCE_DOCS_ROOT exists = True
geopandas = 1.1.3


**Lectura/decisión.** Si `ROOT` no apunta a la raíz del repositorio o `geopandas` no carga, la limpieza espacial no debe continuar. La ruta absoluta impresa ayuda a diagnosticar el entorno local, pero el notebook no depende de esa ruta concreta.


## 1. Catálogo y archivos raw

Se filtra `data_catalog.csv` por las seis fuentes objetivo y se comprueba que cada patrón de `archivo_raw` resuelve a archivos físicos. Esta comprobación sirve para asegurar que la limpieza parte de rutas catalogadas y no de rutas manuales o supuestos locales.

La tabla compacta conserva solo la información necesaria para esta fase: identificador de fuente, formato preferido, ruta raw catalogada, número y nombre de archivos localizados, y ruta interim prevista para la salida limpia.


In [2]:
def relpath(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


def resolve_pattern(pattern: str) -> list[Path]:
    raw_pattern = ROOT / pattern
    matches = [Path(p) for p in glob.glob(str(raw_pattern))]
    return sorted(matches)


catalog = pd.read_csv(CATALOG_PATH)
required_catalog_cols = {
    "dataset_id", "bloque", "prioridad", "nombre_fuente", "source_code", "url_fuente",
    "tipo_acceso", "formato", "formato_preferido", "periodo_dato_objetivo",
    "archivo_raw", "archivo_interim", "unidad_espacial", "granularidad_temporal", "estado",
}
missing_catalog_cols = sorted(required_catalog_cols - set(catalog.columns))
if missing_catalog_cols:
    raise ValueError(f"Faltan columnas obligatorias en data_catalog.csv: {missing_catalog_cols}")

catalog_ser = catalog[catalog["dataset_id"].isin(TARGET_DATASET_IDS)].copy()
missing_dataset_ids = [ds for ds in TARGET_DATASET_IDS if ds not in set(catalog_ser["dataset_id"])]
if missing_dataset_ids:
    raise ValueError(f"Faltan dataset_id en data_catalog.csv: {missing_dataset_ids}")

order = pd.Categorical(catalog_ser["dataset_id"], categories=TARGET_DATASET_IDS, ordered=True)
catalog_ser = catalog_ser.assign(_order=order).sort_values("_order").drop(columns="_order").reset_index(drop=True)

raw_rows = []
RAW_FILES: dict[str, list[Path]] = {}
for row in catalog_ser.itertuples(index=False):
    files = resolve_pattern(row.archivo_raw)
    RAW_FILES[row.dataset_id] = files
    raw_rows.append({
        "dataset_id": row.dataset_id,
        "n_archivos_encontrados": len(files),
        "archivos_encontrados": [relpath(p) for p in files],
    })

raw_check = pd.DataFrame(raw_rows)
missing_raw = raw_check.loc[raw_check["n_archivos_encontrados"].eq(0), "dataset_id"].tolist()
if missing_raw:
    raise FileNotFoundError(f"Hay fuentes sin raw localizado desde data_catalog.csv: {missing_raw}")

catalog_raw_check = (
    catalog_ser[["dataset_id", "formato_preferido", "archivo_raw", "archivo_interim"]]
    .merge(raw_check, on="dataset_id", how="left")
    [["dataset_id", "formato_preferido", "archivo_raw", "n_archivos_encontrados", "archivos_encontrados", "archivo_interim"]]
)
display(catalog_raw_check)


,dataset_id,formato_preferido,archivo_raw,n_archivos_encontrados,archivos_encontrados,archivo_interim
0,ser_zonas,csv,data/raw/ser/ser_zonas/ser_zonas__actual.csv,1,[data/raw/ser/ser_zonas/ser_zonas__actual.csv],data/interim/ser/ser_zonas/ser_zonas_clean.parquet
1,ser_geoportal_limite_ser,geojson,data/raw/ser/ser_geoportal_limite_ser/ser_geoportal_limite_ser.geojson,1,[data/raw/ser/ser_geoportal_limite_ser/ser_geoportal_limite_ser.geojson],data/interim/ser/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet
2,ser_geoportal_barrios_ser,geojson,data/raw/ser/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser.geojson,1,[data/raw/ser/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser.geojson],data/interim/ser/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser_clean.parquet
3,ser_geoportal_bandas_aparcamiento,shp,data/raw/ser/ser_geoportal_bandas_aparcamiento/SHP_ZIP.zip,1,[data/raw/ser/ser_geoportal_bandas_aparcamiento/SHP_ZIP.zip],data/interim/ser/ser_geoportal_bandas_aparcamiento/ser_geoportal_bandas_aparcamiento_clean.parquet
4,ser_calles_plazas,csv,data/raw/ser/ser_calles_plazas/ser_calles_plazas__*.csv,4,"[data/raw/ser/ser_calles_plazas/ser_calles_plazas__2023.csv, data/raw/ser/ser_calles_plazas/ser_calles_plazas__2024.csv, data/raw/ser/ser_calles_plazas/ser_...",data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet
5,ser_parquimetros,"csv,kmz",data/raw/ser/ser_parquimetros/ser_parquimetros__actual.*,2,"[data/raw/ser/ser_parquimetros/ser_parquimetros__actual.csv, data/raw/ser/ser_parquimetros/ser_parquimetros__actual.kmz]",data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet


**Lectura/decisión.** La tabla compacta debe mostrar las seis fuentes actuales y sus archivos localizados desde `data_catalog.csv`. Si falta algún raw, el notebook falla antes de limpiar para evitar rutas manuales o supuestos implícitos.


## 2. Funciones auxiliares tabulares y geoespaciales

Estas funciones auxiliares fijan criterios comunes para todo el notebook antes de limpiar fuentes individuales. Su objetivo es evitar que cada fuente se procese con reglas distintas de lectura, normalización o validación.

Se definen utilidades para normalizar nombres de columnas, limpiar texto, convertir valores numéricos de forma conservadora, leer ficheros tabulares con separadores y codificaciones habituales, cargar capas geoespaciales, comprobar CRS y resumir la calidad geométrica. También se normalizan los colores SER para comparar de forma coherente fuentes que codifican el mismo tipo de plaza con texto o con código RGB.

Estas funciones no agregan información, no construyen joins finales y no generan métricas proxy. Solo preparan una base técnica común para que las limpiezas posteriores sean reproducibles y comparables entre fuentes.


In [3]:
ENCODINGS = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
SEPARATORS = [";", ",", "\t", "|"]
TABULAR_SUFFIXES = {".csv", ".txt", ".xlsx", ".xls"}


def strip_accents(value: str) -> str:
    return "".join(
        char for char in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(char)
    )


COLUMN_ALIASES = {
    "gisx": "gis_x",
    "gis_x": "gis_x",
    "gisy": "gis_y",
    "gis_y": "gis_y",
    "cod_distrito": "cod_distrito",
    "codigo_distrito": "cod_distrito",
    "coddis": "cod_distrito",
    "nomdis": "distrito",
    "distrito": "distrito",
    "cod_barrio": "cod_barrio",
    "codigo_barrio": "cod_barrio",
    "cod_distrito_barrio": "cod_barrio",
    "codbar": "num_barrio",
    "num_barrio": "num_barrio",
    "numero_barrio": "num_barrio",
    "nombar": "barrio",
    "barrio": "barrio",
    "calle": "calle",
    "num_finca": "numero_finca",
    "n_finca": "numero_finca",
    "no_finca": "numero_finca",
    "numero_de_finca": "numero_finca",
    "numero_finca": "numero_finca",
    "num_plazas": "numero_plazas",
    "n_plazas": "numero_plazas",
    "no_plazas": "numero_plazas",
    "numero_de_plazas": "numero_plazas",
    "numero_plazas": "numero_plazas",
    "res_numpla": "numero_plazas",
    "res_numplazas": "numero_plazas",
    "color": "color",
    "bateria_linea": "bateria_linea",
    "bateria_li": "bateria_linea",
    "texto_caje": "texto_cajetin",
    "id": "id_banda",
    "objectid": "objectid",
    "nombre": "nombre",
    "codigo_de_via": "codigo_via",
    "codigo_via": "codigo_via",
    "clase_de_la_via": "clase_via",
    "clase_via": "clase_via",
    "particula_de_la_via": "particula_via",
    "particula_via": "particula_via",
    "nombre_de_la_via": "nombre_via",
    "nombre_via": "nombre_via",
    "tipo_de_tramo": "tipo_tramo",
    "tipo_tramo": "tipo_tramo",
    "nombre_de_la_aproximacion": "nombre_aproximacion",
    "nombre_aproximacion": "nombre_aproximacion",
    "numero_inicial_del_tramo": "numero_inicial",
    "numero_inicial": "numero_inicial",
    "calificador_del_numero_inicial_del_tramo": "calificador_numero_inicial",
    "calificador_numero_inicial": "calificador_numero_inicial",
    "numero_final_del_tramo": "numero_final",
    "numero_final": "numero_final",
    "calificador_del_numero_final_del_tramo": "calificador_numero_final",
    "calificador_numero_final": "calificador_numero_final",
    "zona_s_e_r_del_tramo": "zona_ser_tramo",
    "zona_ser_del_tramo": "zona_ser_tramo",
    "zona_ser_tramo": "zona_ser_tramo",
    "fecha_de_alta": "fecha_de_alta",
    "fecha_alta": "fecha_de_alta",
    "fecha_de_baja": "fecha_de_baja",
    "fecha_baja": "fecha_de_baja",
    "matricula": "matricula",
    "longitud": "longitud",
    "latitud": "latitud",
}


def normalize_key(value: Any) -> str:
    text = str(value).strip().lower()
    text = strip_accents(text)
    text = re.sub(r"\bn[º°]\b", "numero", text)
    text = text.replace("º", " numero ").replace("°", " numero ")
    text = text.replace("/", "_")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")


def normalize_col(col: Any) -> str:
    key = normalize_key(col)
    return COLUMN_ALIASES.get(key, key)


def make_unique_columns(cols: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    result = []
    for col in cols:
        if col not in counts:
            counts[col] = 0
            result.append(col)
        else:
            counts[col] += 1
            result.append(f"{col}_{counts[col]}")
    return result


def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    )


def _normalise_decimal_text(value: Any) -> Any:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().replace("\xa0", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    return text


def to_numeric_series(s: pd.Series) -> pd.Series:
    # Conserva puntos decimales cuando no hay coma decimal; solo elimina puntos como miles si aparece coma.
    text = clean_text_series(s).map(_normalise_decimal_text)
    return pd.to_numeric(text, errors="coerce")


def clean_identifier_series(s: pd.Series) -> pd.Series:
    return clean_text_series(s).str.replace(r"\.0$", "", regex=True)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = make_unique_columns([normalize_col(c) for c in out.columns])
    return out


def normalize_geo_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = gdf.copy()
    geometry_name = out.geometry.name
    renamed = []
    for col in out.columns:
        renamed.append("geometry" if col == geometry_name else normalize_col(col))
    out.columns = make_unique_columns(renamed)
    return gpd.GeoDataFrame(out, geometry="geometry", crs=gdf.crs)


def read_tabular(path: Path, nrows: int | None = None) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Prueba separadores y codificaciones habituales; escoge la lectura con más columnas detectadas.
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, nrows=nrows)
        return df, {"encoding": None, "sep": None, "reader": "read_excel"}
    if suffix not in {".csv", ".txt"}:
        raise ValueError(f"Extension no tabular para lectura base: {path.name}")

    attempts = []
    errors = []
    for encoding in ENCODINGS:
        for sep in SEPARATORS:
            try:
                df = pd.read_csv(path, sep=sep, encoding=encoding, nrows=nrows, low_memory=False)
                attempts.append((df.shape[1], df.shape[0], encoding, sep, df))
            except Exception as exc:
                errors.append({"encoding": encoding, "sep": sep, "error": repr(exc)})
    if not attempts:
        raise ValueError(f"No se pudo leer {path}. Errores iniciales: {errors[:3]}")
    attempts.sort(key=lambda item: (item[0], item[1]), reverse=True)
    n_cols, _, encoding, sep, df = attempts[0]
    return df, {"encoding": encoding, "sep": sep, "reader": "read_csv", "n_cols_detected": n_cols}


def read_geojson(path: Path) -> gpd.GeoDataFrame:
    return gpd.read_file(path)


def read_shp_zip(path: Path) -> gpd.GeoDataFrame:
    with zipfile.ZipFile(path) as zf:
        shp_members = [name for name in zf.namelist() if name.lower().endswith(".shp")]
        if len(shp_members) != 1:
            raise ValueError(f"Se esperaba un unico .shp dentro de {relpath(path)}, encontrados: {shp_members}")
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            shp_path = Path(tmpdir) / shp_members[0]
            return gpd.read_file(shp_path)


def ensure_crs_25830(gdf: gpd.GeoDataFrame, dataset_id: str) -> gpd.GeoDataFrame:
    # Todas las capas Geoportal se trabajan en ETRS89 / UTM zona 30N para áreas, longitudes y distancias en metros.
    if gdf.crs is None:
        raise ValueError(f"{dataset_id} no declara CRS; no se puede asumir EPSG:25830 sin inspeccion manual.")
    epsg = gdf.crs.to_epsg()
    if epsg == 25830:
        return gdf
    return gdf.to_crs(epsg=25830)


def geometry_quality_summary(gdf: gpd.GeoDataFrame, dataset_id: str) -> pd.DataFrame:
    geom = gdf.geometry
    rows = [
        {"dataset_id": dataset_id, "check": "n_filas", "valor": len(gdf)},
        {"dataset_id": dataset_id, "check": "crs", "valor": str(gdf.crs)},
        {"dataset_id": dataset_id, "check": "tipos_geometria", "valor": geom.geom_type.value_counts(dropna=False).to_dict()},
        {"dataset_id": dataset_id, "check": "geometrias_nulas", "valor": int(geom.isna().sum())},
        {"dataset_id": dataset_id, "check": "geometrias_invalidas", "valor": int((~geom.is_valid & geom.notna()).sum())},
    ]
    if len(gdf):
        if geom.geom_type.isin(["Polygon", "MultiPolygon"]).any():
            rows.append({"dataset_id": dataset_id, "check": "area_total_m2_aprox", "valor": float(geom.area.sum())})
        if geom.geom_type.isin(["LineString", "MultiLineString"]).any():
            rows.append({"dataset_id": dataset_id, "check": "longitud_total_m_aprox", "valor": float(geom.length.sum())})
    return pd.DataFrame(rows)


def extract_year_from_name(path: Path) -> int | None:
    years = re.findall(r"(20\d{2})", path.name)
    return int(years[-1]) if years else None


def split_code_text(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    text = clean_text_series(series)
    extracted = text.str.extract(r"^\s*([0-9]+)\s*[-\.]?\s*(.*)$")
    code = pd.to_numeric(extracted[0], errors="coerce").astype("Int64")
    name = clean_text_series(extracted[1]).where(extracted[1].notna(), text)
    name = name.mask(name.isna(), text)
    return code, name


def parse_barrio_text(series: pd.Series) -> pd.DataFrame:
    text = clean_text_series(series)
    pattern = text.str.extract(r"^\s*(\d{1,2})\s*[-/]\s*(\d{1,2})\s+(.+)$")
    return pd.DataFrame({
        "cod_distrito_from_barrio": pd.to_numeric(pattern[0], errors="coerce").astype("Int64"),
        "num_barrio_from_barrio": pd.to_numeric(pattern[1], errors="coerce").astype("Int64"),
        "barrio_nombre_from_barrio": clean_text_series(pattern[2]),
    })


def compose_barrio_code(cod_distrito: pd.Series, num_barrio: pd.Series) -> pd.Series:
    cod_distrito_num = pd.to_numeric(cod_distrito, errors="coerce")
    num_barrio_num = pd.to_numeric(num_barrio, errors="coerce")
    return (cod_distrito_num * 100 + num_barrio_num).round().astype("Int64")


def normalize_label(value: Any) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = strip_accents(str(value)).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


SER_COLOR_ALIASES = {
    "043000255 azul": "azul",
    "077214010 verde": "verde",
    "081209246 alta rotacion": "alta rotacion",
    "255000000 rojo": "rojo",
    "255140000 naranja": "naranja",
    "azul": "azul",
    "verde": "verde",
    "alta rotacion": "alta rotacion",
    "rojo": "rojo",
    "naranja": "naranja",
    "gris": "gris",
}


def normalize_ser_color(value: Any) -> str | pd.NA:
    # Unifica colores escritos como texto y como código RGB + texto para comparaciones entre fuentes SER.
    label = normalize_label(value)
    if pd.isna(label):
        return pd.NA
    return SER_COLOR_ALIASES.get(label, label)


def color_for_clean(value: Any) -> str | pd.NA:
    color = normalize_ser_color(value)
    if pd.isna(color):
        return pd.NA
    return str(color).replace(" ", "_")


def normalize_street_name(value: Any) -> str | pd.NA:
    label = normalize_label(value)
    if pd.isna(label):
        return pd.NA
    label = re.sub(r"[^a-z0-9 ]+", " ", label)
    label = re.sub(r"\b(calle|callejon|avenida|avda|paseo|plaza|plz|ronda|glorieta|via)\b", " ", label)
    return re.sub(r"\s+", " ", label).strip()


def union_geometry(gdf: gpd.GeoDataFrame):
    return gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union


def ensure_parquet_engine() -> None:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError("Para escribir Parquet instala pyarrow en el entorno activo.") from exc

print("Funciones auxiliares cargadas")


Funciones auxiliares cargadas


## 3. Inspección estructural global

Se carga cada raw en `RAW_TABLES` o `GEO_RAW` y se deja un control inicial compacto. Esta inspección global no decide todavía qué columnas se conservan: solo confirma que todas las fuentes cargan, qué tamaño tienen y qué columnas normalizadas se han detectado.

La decisión de conservar, descartar o diagnosticar variables se realiza dentro de cada bloque de fuente, donde el contexto de uso es más claro.


In [4]:
RAW_TABLES: dict[str, list[dict[str, Any]]] = {}
inspection_rows = []

for dataset_id in ["ser_zonas", "ser_calles_plazas", "ser_parquimetros"]:
    items = []
    for path in RAW_FILES[dataset_id]:
        if path.suffix.lower() not in TABULAR_SUFFIXES:
            continue
        df, meta = read_tabular(path)
        norm = normalize_columns(df)
        items.append({"path": path, "raw": df, "norm": norm, "meta": meta})
        inspection_rows.append({
            "dataset_id": dataset_id,
            "archivo": relpath(path),
            "shape": df.shape,
            "columnas_normalizadas": list(norm.columns),
        })
    if not items:
        raise ValueError(f"No hay archivos tabulares legibles para {dataset_id}")
    RAW_TABLES[dataset_id] = items

GEO_RAW: dict[str, gpd.GeoDataFrame] = {}
for dataset_id in ["ser_geoportal_limite_ser", "ser_geoportal_barrios_ser", "ser_geoportal_bandas_aparcamiento"]:
    path = RAW_FILES[dataset_id][0]
    gdf = read_shp_zip(path) if path.suffix.lower() == ".zip" else read_geojson(path)
    gdf = ensure_crs_25830(gdf, dataset_id)
    gdf = normalize_geo_columns(gdf)
    GEO_RAW[dataset_id] = gdf
    inspection_rows.append({
        "dataset_id": dataset_id,
        "archivo": relpath(path),
        "shape": gdf.shape,
        "columnas_normalizadas": list(gdf.columns),
    })

inspection = pd.DataFrame(inspection_rows)
global_inspection = (
    inspection.groupby("dataset_id", sort=False)
    .agg(
        n_archivos=("archivo", "nunique"),
        shape=("shape", list),
        columnas_normalizadas=("columnas_normalizadas", list),
    )
    .reset_index()
)
display(global_inspection)


,dataset_id,n_archivos,shape,columnas_normalizadas
0,ser_zonas,1,"[(6720, 11)]","[[codigo_via, clase_via, particula_via, nombre_via, tipo_tramo, nombre_aproximacion, numero_inicial, calificador_numero_inicial, numero_final, calificador_n..."
1,ser_calles_plazas,4,"[(32111, 9), (33700, 12), (34583, 12), (34520, 12)]","[[gis_x, gis_y, distrito, barrio, calle, numero_finca, color, bateria_linea, numero_plazas], [gis_x, gis_y, cod_distrito, distrito, cod_barrio, num_barrio, ..."
2,ser_parquimetros,1,"[(6246, 14)]","[[gis_x, gis_y, fecha_de_alta, fecha_de_baja, cod_distrito, distrito, cod_barrio, num_barrio, barrio, calle, numero_finca, matricula, longitud, latitud]]"
3,ser_geoportal_limite_ser,1,"[(1, 3)]","[[nombre, objectid, geometry]]"
4,ser_geoportal_barrios_ser,1,"[(67, 6)]","[[cod_distrito, distrito, num_barrio, barrio, objectid, geometry]]"
5,ser_geoportal_bandas_aparcamiento,1,"[(87615, 6)]","[[id_banda, color, bateria_linea, numero_plazas, texto_cajetin, geometry]]"


**Lectura/decisión.** Este control confirma que las seis fuentes cargan y que sus columnas normalizadas son visibles. La decisión de conservar, descartar o diagnosticar variables se documenta dentro de cada bloque de limpieza.


## 4. Limpieza de `ser_zonas`

**Qué mide.** `ser_zonas` describe la correspondencia entre una vía municipal, un rango de numeración, la paridad/lado del tramo y la zona SER asociada.

**Uso en el TFM.** Se mantiene como lookup auxiliar para validaciones futuras por finca, tramo o dirección. No se usa para pintar el mapa, no mide capacidad y no se cruza todavía con tiques.

**Columnas conservadas.** Se conservan `codigo_via`, `clase_via`, `particula_via`, `nombre_via`, `tipo_tramo`, `nombre_aproximacion`, `numero_inicial`, `numero_final` y `zona_ser_tramo`, porque definen la lógica vía/rango/paridad → zona SER.

**Columnas descartadas.** No se conservan flags temporales ni trazabilidad de archivo en el clean final. También se descartan `calificador_numero_inicial` y `calificador_numero_final`: se diagnostican porque podrían describir casos de numeración especial, pero su cobertura es residual frente al tamaño de la fuente y no compensa mantenerlas en la salida limpia de esta fase.

**Validaciones.** Se comprueban nulos en campos clave, cobertura de calificadores y ausencia de tramos sin rango/paridad suficiente o sin zona SER. Si estos checks fallasen, la fuente no debería usarse como lookup auxiliar sin revisión manual posterior.


In [5]:
SER_ZONAS_REQUIRED_COLUMNS = [
    "codigo_via", "clase_via", "particula_via", "nombre_via", "tipo_tramo",
    "nombre_aproximacion", "numero_inicial", "numero_final", "zona_ser_tramo",
]
SER_ZONAS_CALIFICADOR_COLUMNS = ["calificador_numero_inicial", "calificador_numero_final"]
SER_ZONAS_INPUT_COLUMNS = [
    "codigo_via", "clase_via", "particula_via", "nombre_via", "tipo_tramo",
    "nombre_aproximacion", "numero_inicial", "calificador_numero_inicial",
    "numero_final", "calificador_numero_final", "zona_ser_tramo",
]
SER_ZONAS_FINAL_COLUMNS = SER_ZONAS_REQUIRED_COLUMNS.copy()


def clean_ser_zonas(items: list[dict[str, Any]]) -> pd.DataFrame:
    parts = []
    for item in items:
        df = item["norm"].copy()
        for col in SER_ZONAS_INPUT_COLUMNS:
            if col not in df.columns:
                df[col] = pd.NA
        out = pd.DataFrame({
            "codigo_via": pd.to_numeric(df["codigo_via"], errors="coerce").astype("Int64"),
            "clase_via": clean_text_series(df["clase_via"]),
            "particula_via": clean_text_series(df["particula_via"]),
            "nombre_via": clean_text_series(df["nombre_via"]),
            "tipo_tramo": clean_text_series(df["tipo_tramo"]),
            "nombre_aproximacion": clean_text_series(df["nombre_aproximacion"]),
            "numero_inicial": pd.to_numeric(df["numero_inicial"], errors="coerce").astype("Int64"),
            "calificador_numero_inicial": clean_text_series(df["calificador_numero_inicial"]),
            "numero_final": pd.to_numeric(df["numero_final"], errors="coerce").astype("Int64"),
            "calificador_numero_final": clean_text_series(df["calificador_numero_final"]),
            "zona_ser_tramo": pd.to_numeric(df["zona_ser_tramo"], errors="coerce").astype("Int64"),
        })
        out["flag_tramo_ambiguo"] = out[["tipo_tramo", "numero_inicial", "numero_final"]].isna().any(axis=1)
        out["flag_sin_zona_ser"] = out["zona_ser_tramo"].isna()
        parts.append(out)
    return pd.concat(parts, ignore_index=True)


ser_zonas_diagnostic = clean_ser_zonas(RAW_TABLES["ser_zonas"])

zonas_null_checks = ["codigo_via", "nombre_via", "tipo_tramo", "numero_inicial", "numero_final", "zona_ser_tramo"]
calificadores_no_nulos = {
    col: int(ser_zonas_diagnostic[col].notna().sum()) for col in SER_ZONAS_CALIFICADOR_COLUMNS
}
calificadores_pct_no_nulos = {
    col: round(calificadores_no_nulos[col] / len(ser_zonas_diagnostic) * 100, 3)
    for col in SER_ZONAS_CALIFICADOR_COLUMNS
}

zonas_quality_rows = [
    ("n_filas", int(len(ser_zonas_diagnostic)), "Tramos viales recibidos tras normalizar columnas."),
    *[(f"n_{col}_nulo", int(ser_zonas_diagnostic[col].isna().sum()), f"Campos nulos en {col}.") for col in zonas_null_checks],
    ("n_calificador_numero_inicial_no_nulo", calificadores_no_nulos["calificador_numero_inicial"], "Valores informados en calificador inicial."),
    ("n_calificador_numero_final_no_nulo", calificadores_no_nulos["calificador_numero_final"], "Valores informados en calificador final."),
    ("pct_calificador_numero_inicial_no_nulo", calificadores_pct_no_nulos["calificador_numero_inicial"], "Cobertura porcentual del calificador inicial; se descarta por cobertura residual."),
    ("pct_calificador_numero_final_no_nulo", calificadores_pct_no_nulos["calificador_numero_final"], "Cobertura porcentual del calificador final; se descarta por cobertura residual."),
    ("n_tramo_ambiguo", int(ser_zonas_diagnostic["flag_tramo_ambiguo"].sum()), "Tramos sin tipo/rango suficiente para uso futuro por finca o paridad."),
    ("n_sin_zona_ser", int(ser_zonas_diagnostic["flag_sin_zona_ser"].sum()), "Tramos sin zona SER asociada."),
]
zonas_quality = pd.DataFrame(zonas_quality_rows, columns=["check", "valor", "interpretacion"])

display(zonas_quality)


,check,valor,interpretacion
0,n_filas,6720.000,Tramos viales recibidos tras normalizar columnas.
1,n_codigo_via_nulo,0.000,Campos nulos en codigo_via.
2,n_nombre_via_nulo,0.000,Campos nulos en nombre_via.
3,n_tipo_tramo_nulo,0.000,Campos nulos en tipo_tramo.
4,n_numero_inicial_nulo,0.000,Campos nulos en numero_inicial.
5,n_numero_final_nulo,0.000,Campos nulos en numero_final.
6,n_zona_ser_tramo_nulo,0.000,Campos nulos en zona_ser_tramo.
7,n_calificador_numero_inicial_no_nulo,3.000,Valores informados en calificador inicial.
8,n_calificador_numero_final_no_nulo,39.000,Valores informados en calificador final.
9,pct_calificador_numero_inicial_no_nulo,0.045,Cobertura porcentual del calificador inicial; se descarta por cobertura residual.


**Lectura/decisión.** Los checks muestran que los campos mínimos de `ser_zonas` no presentan nulos en `codigo_via`, `nombre_via`, `tipo_tramo`, `numero_inicial`, `numero_final` ni `zona_ser_tramo`. Tampoco aparecen tramos ambiguos ni tramos sin zona SER.

Los calificadores de numeración sí tienen algún valor informado, pero su cobertura es residual: 3 registros en `calificador_numero_inicial` y 39 registros en `calificador_numero_final` sobre 6.720 tramos. En esta fase se descartan del clean final porque no aportan una variable operativa estable para el uso previsto del notebook. Si más adelante se construye una lógica fina por dirección o numeración especial, esos campos deberán revisarse de nuevo desde el raw.

Con esta evidencia, `ser_zonas` queda como lookup auxiliar limpio. Se eliminan del output los flags de diagnóstico y los calificadores de cobertura residual; su función en este notebook era comprobar estructura y posible utilidad, no viajar a la salida limpia.


In [6]:
ser_zonas_clean = ser_zonas_diagnostic[SER_ZONAS_FINAL_COLUMNS].copy()
display(ser_zonas_clean.head())


,codigo_via,clase_via,particula_via,nombre_via,tipo_tramo,nombre_aproximacion,numero_inicial,numero_final,zona_ser_tramo
0,200,CALLE,DE LA,ABADA,impar,NUM,1,999,163
1,200,CALLE,DE LA,ABADA,par,NUM,2,998,163
2,300,CALLE,DE LOS,ABADES,impar,NUM,1,999,12
3,300,CALLE,DE LOS,ABADES,par,NUM,2,998,12
4,400,CALLE,DE LA,ABADESA,impar,NUM,1,999,65


## 5. Limpieza de `ser_geoportal_limite_ser`

**Qué mide.** `ser_geoportal_limite_ser` contiene el polígono oficial del ámbito del Servicio de Estacionamiento Regulado.

**Uso en el TFM.** Sirve como geometría de referencia para validar si puntos o líneas SER caen dentro del área regulada y como base espacial para mapas posteriores.

**Columnas conservadas.** Se conservan `objectid`, `nombre` y `geometry`, porque identifican el polígono y su geometría oficial.

**Columnas descartadas.** No se añaden `dataset_id` ni `archivo_origen` al clean final, porque la fuente ya queda trazada por `data_catalog.csv` y por la ruta de salida.

**Validaciones.** Se comprueba número de geometrías, CRS, validez geométrica, geometrías nulas, área aproximada y nombres únicos. Esta fuente no requiere limpieza pesada: requiere validación y normalización geoespacial.


In [7]:
LIMITE_FINAL_COLUMNS = ["objectid", "nombre", "geometry"]


def clean_limite_ser(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    df = gdf.copy()
    for col in ["objectid", "nombre"]:
        if col not in df.columns:
            df[col] = pd.NA
    out = gpd.GeoDataFrame({
        "objectid": pd.to_numeric(df["objectid"], errors="coerce").astype("Int64"),
        "nombre": clean_text_series(df["nombre"]),
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    return ensure_crs_25830(out, "ser_geoportal_limite_ser")


ser_geoportal_limite_ser_clean = clean_limite_ser(GEO_RAW["ser_geoportal_limite_ser"])
limite_geom_valid = ser_geoportal_limite_ser_clean.geometry.is_valid
limite_quality = pd.DataFrame([
    ("n_geometrias", int(len(ser_geoportal_limite_ser_clean)), "Número de geometrías del límite SER."),
    ("crs_epsg", int(ser_geoportal_limite_ser_clean.crs.to_epsg()), "Debe ser 25830 para medir distancias y áreas en metros."),
    ("n_geometrias_validas", int(limite_geom_valid.sum()), "Geometrías válidas."),
    ("n_geometrias_nulas", int(ser_geoportal_limite_ser_clean.geometry.isna().sum()), "Geometrías nulas."),
    ("area_aproximada_m2", float(ser_geoportal_limite_ser_clean.geometry.area.sum()), "Área total aproximada en m2."),
    ("nombres_unicos", ser_geoportal_limite_ser_clean["nombre"].dropna().unique().tolist(), "Nombres únicos de la capa."),
], columns=["check", "valor", "interpretacion"])

display(limite_quality)


,check,valor,interpretacion
0,n_geometrias,1,Número de geometrías del límite SER.
1,crs_epsg,25830,Debe ser 25830 para medir distancias y áreas en metros.
2,n_geometrias_validas,1,Geometrías válidas.
3,n_geometrias_nulas,0,Geometrías nulas.
4,area_aproximada_m2,58686575.954764,Área total aproximada en m2.
5,nombres_unicos,[Zona S.E.R.],Nombres únicos de la capa.


**Lectura/decisión.** La capa contiene una única geometría válida, sin geometrías nulas, en EPSG:25830. El área aproximada es de 58.686.575,95 m² y el nombre único es `Zona S.E.R.`.

Con esta evidencia, el límite SER se acepta como geometría oficial de referencia. No se muestra una tabla limpia adicional porque el output final tiene una sola fila y su contenido ya queda suficientemente descrito por los checks.


## 6. Limpieza de `ser_geoportal_barrios_ser`

**Qué mide.** `ser_geoportal_barrios_ser` contiene los polígonos de barrios dentro del ámbito SER.

**Uso en el TFM.** Permite dividir el mapa SER por barrios y habilita futuras agregaciones espaciales por barrio. No sustituye a `ser_calles_plazas` como fuente de capacidad.

**Columnas conservadas.** Se conservan `cod_distrito`, `distrito`, `num_barrio`, `cod_barrio`, `barrio`, `objectid` y `geometry`. La regla `cod_barrio = cod_distrito * 100 + num_barrio` se usa como armonización del código compuesto de barrio.

**Columnas descartadas.** Se excluye del clean el polígono “No está en la zona SER”, porque no representa un barrio SER utilizable. También se descartan flags temporales y trazabilidad redundante.

**Validaciones.** Se comprueban conteos de polígonos, áreas, coherencia entre la unión de barrios y el límite SER, y duplicados de `cod_barrio`. Si aparece un duplicado, se diagnostica geométricamente antes de decidir si se elimina, se conserva o se pospone su disolución.


In [8]:
BARRIOS_FINAL_COLUMNS = ["cod_distrito", "distrito", "num_barrio", "cod_barrio", "barrio", "objectid", "geometry"]
TOL_AREA_M2 = 1.0


def clean_barrios_ser_diagnostic(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    df = gdf.copy()
    for col in ["cod_distrito", "distrito", "num_barrio", "cod_barrio", "barrio", "objectid"]:
        if col not in df.columns:
            df[col] = pd.NA
    cod_distrito = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
    num_barrio = pd.to_numeric(df["num_barrio"], errors="coerce").astype("Int64")
    cod_barrio = compose_barrio_code(cod_distrito, num_barrio)
    barrio = clean_text_series(df["barrio"])
    barrio_norm = barrio.map(normalize_label)
    flag_en_zona_ser = barrio_norm.ne("no esta en la zona ser") & cod_barrio.notna()
    diagnostic = gpd.GeoDataFrame({
        "cod_distrito": cod_distrito,
        "distrito": clean_text_series(df["distrito"]),
        "num_barrio": num_barrio,
        "cod_barrio": cod_barrio,
        "barrio": barrio,
        "objectid": pd.to_numeric(df["objectid"], errors="coerce").astype("Int64"),
        "flag_en_zona_ser": flag_en_zona_ser.fillna(False),
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    return ensure_crs_25830(diagnostic, "ser_geoportal_barrios_ser")


def fmt_int(value: int) -> int:
    return int(value)


def fmt_m2(value: float) -> str:
    return f"{float(value):.3f}"


def fmt_pct(value: float) -> str:
    return f"{float(value):.9f}"


ser_geoportal_barrios_ser_diagnostic = clean_barrios_ser_diagnostic(GEO_RAW["ser_geoportal_barrios_ser"])
barrios_en_zona = ser_geoportal_barrios_ser_diagnostic["flag_en_zona_ser"]
barrios_candidate_clean = ser_geoportal_barrios_ser_diagnostic.loc[barrios_en_zona, BARRIOS_FINAL_COLUMNS].copy()

barrios_union_geom = union_geometry(barrios_candidate_clean)
limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
area_union_barrios_ser_m2 = float(barrios_union_geom.area)
area_limite_ser_m2 = float(limite_geom.area)
diferencia_area_m2 = area_union_barrios_ser_m2 - area_limite_ser_m2
diferencia_area_pct = float(diferencia_area_m2 / area_limite_ser_m2 * 100) if area_limite_ser_m2 else np.nan

barrios_quality = pd.DataFrame([
    ("n_poligonos_total", fmt_int(len(ser_geoportal_barrios_ser_diagnostic)), "Polígonos recibidos en raw."),
    ("n_poligonos_en_zona_ser", fmt_int(barrios_en_zona.sum()), "Polígonos SER reales que pasan a candidato clean."),
    ("n_poligonos_no_ser", fmt_int((~barrios_en_zona).sum()), "Polígonos no SER excluidos del clean."),
    ("area_total_m2", fmt_m2(ser_geoportal_barrios_ser_diagnostic.geometry.area.sum()), "Área total del raw."),
    ("area_en_zona_ser_m2", fmt_m2(ser_geoportal_barrios_ser_diagnostic.loc[barrios_en_zona].geometry.area.sum()), "Área sumada de polígonos SER conservados."),
    ("area_union_barrios_ser_m2", fmt_m2(area_union_barrios_ser_m2), "Área de la unión geométrica de barrios SER limpios."),
    ("area_limite_ser_m2", fmt_m2(area_limite_ser_m2), "Área del límite SER oficial."),
    ("diferencia_area_m2", fmt_m2(diferencia_area_m2), "Diferencia unión barrios SER menos límite SER."),
    ("diferencia_area_pct", fmt_pct(diferencia_area_pct), "Diferencia relativa sobre área del límite SER."),
    ("duplicados_cod_barrio_en_zona_ser", fmt_int(barrios_candidate_clean.duplicated("cod_barrio").sum()), "Duplicados de código compuesto dentro de zona SER."),
], columns=["check", "valor", "interpretacion"])

display(barrios_quality)


,check,valor,interpretacion
0,n_poligonos_total,67,Polígonos recibidos en raw.
1,n_poligonos_en_zona_ser,66,Polígonos SER reales que pasan a candidato clean.
2,n_poligonos_no_ser,1,Polígonos no SER excluidos del clean.
3,area_total_m2,604455106.869,Área total del raw.
4,area_en_zona_ser_m2,58686575.971,Área sumada de polígonos SER conservados.
5,area_union_barrios_ser_m2,58686575.949,Área de la unión geométrica de barrios SER limpios.
6,area_limite_ser_m2,58686575.955,Área del límite SER oficial.
7,diferencia_area_m2,-0.005,Diferencia unión barrios SER menos límite SER.
8,diferencia_area_pct,-0.000000009,Diferencia relativa sobre área del límite SER.
9,duplicados_cod_barrio_en_zona_ser,1,Duplicados de código compuesto dentro de zona SER.


**Lectura/decisión.** El raw contiene 67 polígonos, de los cuales 66 corresponden a barrios SER reales y 1 corresponde a “No está en la zona SER”. Ese polígono no-SER se excluye del clean porque inflaría el área total y no debe intervenir en mapas ni agregaciones SER.

La unión geométrica de los barrios SER limpios tiene un área de 58.686.575,949 m², prácticamente idéntica al límite SER oficial, con una diferencia de -0,005 m² (-0,000000009 %). Esto valida que, tras excluir el polígono no-SER, la cobertura espacial de barrios SER cuadra con el límite oficial.

Se detecta un duplicado de `cod_barrio` dentro de zona SER, por lo que se realiza un diagnóstico específico antes de cerrar el clean.


### 6.1. Diagnóstico de duplicados de `cod_barrio`

El duplicado detectado corresponde al código `904`, asociado a `Valdezarza` y `Valdezarza Fase III`. Esta revisión comprueba si el duplicado implica un error geométrico real o si se trata de dos piezas territoriales con el mismo código compuesto.

Se comprueba: si las geometrías se tocan o intersectan, si existe solape relevante, si una contiene a la otra, si son disjuntas, y si la unión queda dentro del límite SER.


In [9]:
duplicados_barrios_ser = (
    barrios_candidate_clean
    .assign(area_m2=lambda df: df.geometry.area)
    .loc[lambda df: df.duplicated("cod_barrio", keep=False)]
    .sort_values(["cod_barrio", "barrio"])
)

if not duplicados_barrios_ser.empty:
    display(duplicados_barrios_ser.drop(columns="geometry"))

barrios_904 = barrios_candidate_clean.loc[barrios_candidate_clean["cod_barrio"].eq(904)].copy()
if len(barrios_904) == 2:
    geom_a, geom_b = barrios_904.geometry.iloc[0], barrios_904.geometry.iloc[1]
    union_904 = geom_a.union(geom_b)
    area_interseccion_m2 = float(geom_a.intersection(geom_b).area)
    cod_904_geometria = pd.DataFrame([{
        "cod_barrio": 904,
        "barrio_a": barrios_904["barrio"].iloc[0],
        "barrio_b": barrios_904["barrio"].iloc[1],
        "area_a_m2": float(geom_a.area),
        "area_b_m2": float(geom_b.area),
        "se_tocan_o_intersectan": bool(geom_a.intersects(geom_b)),
        "area_interseccion_m2": area_interseccion_m2,
        "solape_relevante": bool(area_interseccion_m2 > TOL_AREA_M2),
        "a_contiene_b": bool(geom_a.contains(geom_b)),
        "b_contiene_a": bool(geom_b.contains(geom_a)),
        "son_disjuntas": bool(geom_a.disjoint(geom_b)),
        "area_union_m2": float(union_904.area),
        "area_union_dentro_limite_ser_m2": float(union_904.intersection(limite_geom).area),
        "diferencia_union_vs_limite_intersec_m2": float(union_904.area - union_904.intersection(limite_geom).area),
    }])
elif len(barrios_904) > 0:
    cod_904_geometria = pd.DataFrame([{
        "cod_barrio": 904,
        "n_poligonos": int(len(barrios_904)),
        "nota": "No hay exactamente dos polígonos con cod_barrio 904; revisar manualmente si cambia el raw.",
    }])
else:
    cod_904_geometria = pd.DataFrame()

if not cod_904_geometria.empty:
    display(cod_904_geometria)


,cod_distrito,distrito,num_barrio,cod_barrio,barrio,objectid,area_m2
46,9,Moncloa - Aravaca,4,904,Valdezarza,47,359529.629332
65,9,Moncloa - Aravaca,4,904,Valdezarza Fase III,67,209555.978181


,cod_barrio,barrio_a,barrio_b,area_a_m2,area_b_m2,se_tocan_o_intersectan,area_interseccion_m2,solape_relevante,a_contiene_b,b_contiene_a,son_disjuntas,area_union_m2,area_union_dentro_limite_ser_m2,diferencia_union_vs_limite_intersec_m2
0,904,Valdezarza,Valdezarza Fase III,359529.629332,209555.978181,True,0.002087,False,False,False,False,569085.605426,569085.604794,0.000631


**Lectura/decisión.** El duplicado `904` tiene dos geometrías con áreas de 359.529,63 m² y 209.555,98 m². Aunque `se_tocan_o_intersectan = True`, el área de intersección es de solo 0,002087 m², por debajo de la tolerancia de 1 m²; por tanto, `solape_relevante = False`.

Ninguna geometría contiene a la otra y la diferencia entre el área de la unión y su intersección con el límite SER es residual. La decisión es conservar ambas geometrías en el clean, sin disolver ni eliminar. Si más adelante se necesita un único polígono por `cod_barrio`, la disolución se hará en un notebook posterior y deberá documentarse explícitamente.


In [10]:
ser_geoportal_barrios_ser_clean = barrios_candidate_clean.copy()
display(ser_geoportal_barrios_ser_clean.head())


,cod_distrito,distrito,num_barrio,cod_barrio,barrio,objectid,geometry
0,2,Arganzuela,4,204,Legazpi,1,"POLYGON ((442590.498 4471523.292, 442320.51 4471837.852, 442297.494 4471826.497, 442293.234 4471824.097, 442277.704 4471816.638, 442236.044 4471796.628, 442..."
1,2,Arganzuela,3,203,Chopera,2,"POLYGON ((441005.241 4471403.949, 441029.761 4471621.159, 441038.641 4471744.549, 440943.022 4471928.53, 440917.233 4471982.2, 440876.633 4472057.501, 44077..."
2,2,Arganzuela,5,205,Delicias,3,"POLYGON ((441830.29 4472424.757, 441364.831 4472462.098, 441248.722 4472471.539, 441239.782 4472472.269, 441220.092 4472473.589, 441122.613 4472480.14, 4411..."
3,3,Retiro,2,302,Adelfas,4,"POLYGON ((443555.629 4473070.991, 443236.081 4473104.795, 443131.282 4473109.547, 443129.162 4473095.287, 443126.872 4473079.917, 443121.082 4473066.397, 44..."
4,2,Arganzuela,2,202,Acacias,5,"POLYGON ((440385.948 4472954.316, 440350.318 4472954.016, 440272.919 4472969.567, 440050.9 4473024.689, 440000.953 4473038.581, 439890.401 4473069.33, 43979..."


## 7. Limpieza de `ser_geoportal_bandas_aparcamiento`

**Qué mide.** `ser_geoportal_bandas_aparcamiento` contiene la geometría lineal de las bandas de aparcamiento SER. Es la fuente que permite representar en un mapa las líneas coloreadas de plazas reguladas.

**Uso en el TFM.** Se mantiene como capa cartográfica de oferta física regulada y como contraste espacial frente a `ser_calles_plazas`. No es el denominador principal del modelo: la capacidad tabular histórica sigue viniendo de `ser_calles_plazas`.

**Columnas conservadas.** Se conservan `id_banda`, `color`, `numero_plazas` y `geometry`, porque identifican la banda, el tipo de plaza, el número de plazas y la línea que se dibujará.

**Columnas descartadas.** Se descartan `texto_cajetin`, `bateria_linea`, `longitud_m`, flags y trazabilidad de origen. `longitud_m` no se guarda porque puede recalcularse desde la geometría si se necesita en un análisis posterior.

**Validaciones.** Se comprueban colores regulados, bandas grises, plazas nulas/cero/negativas, geometrías inválidas y posición respecto al límite SER. La posición espacial se evalúa con criterio estricto y con un buffer de 5 m: el buffer evita sobrerreaccionar ante líneas situadas en el borde del polígono o desplazadas levemente por precisión cartográfica.

En esta limpieza interim no se eliminan bandas reguladas solo por quedar fuera del límite con buffer. Si tienen color SER válido, plazas informadas y geometría válida, se conservan y la incidencia espacial queda diagnosticada para el notebook posterior de mapas.


In [11]:
BANDAS_FINAL_COLUMNS = ["id_banda", "color", "numero_plazas", "geometry"]


def diagnose_bandas_aparcamiento(gdf: gpd.GeoDataFrame, limite_geom) -> gpd.GeoDataFrame:
    df = gdf.copy()
    for col in ["id_banda", "color", "bateria_linea", "numero_plazas", "texto_cajetin"]:
        if col not in df.columns:
            df[col] = pd.NA
    color_norm = clean_text_series(df["color"]).map(normalize_ser_color)
    numero_plazas = to_numeric_series(df["numero_plazas"]).round().astype("Int64")
    limite_geom_buffer_5m_local = limite_geom.buffer(5)
    diagnostic = gpd.GeoDataFrame({
        "id_banda": pd.to_numeric(df["id_banda"], errors="coerce").astype("Int64"),
        "color": color_norm.map(lambda x: str(x).replace(" ", "_") if pd.notna(x) else pd.NA).astype("string"),
        "color_norm_diagnostico": color_norm,
        "bateria_linea": clean_text_series(df["bateria_linea"]),
        "numero_plazas": numero_plazas,
        "texto_cajetin": clean_text_series(df["texto_cajetin"]),
        "longitud_m": df.geometry.length,
        "geometry": df.geometry,
    }, geometry="geometry", crs=df.crs)
    diagnostic = ensure_crs_25830(diagnostic, "ser_geoportal_bandas_aparcamiento")
    diagnostic["flag_color_gris"] = diagnostic["color_norm_diagnostico"].eq("gris").fillna(False)
    diagnostic["flag_color_ser_regulado"] = diagnostic["color_norm_diagnostico"].isin(SER_REGULATED_COLORS_NORM).fillna(False)
    diagnostic["flag_plazas_nulas"] = diagnostic["numero_plazas"].isna()
    diagnostic["flag_plazas_cero"] = diagnostic["numero_plazas"].fillna(-1).eq(0)
    diagnostic["flag_plazas_negativas"] = diagnostic["numero_plazas"].fillna(0).lt(0)
    diagnostic["flag_geom_invalida"] = ~diagnostic.geometry.is_valid | diagnostic.geometry.isna()
    diagnostic["flag_fuera_limite_estricto"] = ~diagnostic.geometry.intersects(limite_geom)
    diagnostic["flag_fuera_limite_buffer_5m"] = ~diagnostic.geometry.intersects(limite_geom_buffer_5m_local)
    return diagnostic


limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
ser_geoportal_bandas_aparcamiento_diagnostic = diagnose_bandas_aparcamiento(
    GEO_RAW["ser_geoportal_bandas_aparcamiento"], limite_geom
)
bandas_diag = ser_geoportal_bandas_aparcamiento_diagnostic
bandas_keep = (
    bandas_diag["flag_color_ser_regulado"]
    & bandas_diag["numero_plazas"].notna()
    & ~bandas_diag["flag_geom_invalida"]
)
bandas_quality = pd.DataFrame([
    ("n_bandas_raw", int(len(bandas_diag)), "Bandas recibidas."),
    ("n_bandas_candidato_clean", int(bandas_keep.sum()), "Bandas SER reguladas válidas por color, plazas y geometría; el límite queda como diagnóstico."),
    ("n_bandas_sin_color", int(bandas_diag["color_norm_diagnostico"].isna().sum()), "Bandas sin color normalizable; se excluyen del clean."),
    ("n_bandas_gris", int(bandas_diag["flag_color_gris"].sum()), "Bandas grises diagnosticadas; se excluyen por no ser color SER regulado objetivo."),
    ("plazas_gris", int(bandas_diag.loc[bandas_diag["flag_color_gris"], "numero_plazas"].fillna(0).sum()), "Plazas asociadas a bandas grises."),
    ("n_plazas_nulas", int(bandas_diag["flag_plazas_nulas"].sum()), "Registros con numero_plazas nulo; se excluyen del clean."),
    ("n_plazas_cero", int(bandas_diag["flag_plazas_cero"].sum()), "Registros con cero plazas."),
    ("n_plazas_negativas", int(bandas_diag["flag_plazas_negativas"].sum()), "Registros con plazas negativas."),
    ("n_geometrias_invalidas", int(bandas_diag["flag_geom_invalida"].sum()), "Geometrías nulas o inválidas."),
    ("n_bandas_gris_fuera_limite_estricto", int((bandas_diag["flag_color_gris"] & bandas_diag["flag_fuera_limite_estricto"]).sum()), "Bandas grises fuera del límite estricto."),
    ("n_bandas_gris_fuera_limite_buffer_5m", int((bandas_diag["flag_color_gris"] & bandas_diag["flag_fuera_limite_buffer_5m"]).sum()), "Bandas grises fuera del límite con buffer 5 m."),
    ("n_bandas_reguladas_fuera_limite_estricto", int((bandas_diag["flag_color_ser_regulado"] & bandas_diag["flag_fuera_limite_estricto"]).sum()), "Bandas reguladas fuera del límite estricto."),
    ("n_bandas_reguladas_fuera_limite_buffer_5m", int((bandas_diag["flag_color_ser_regulado"] & bandas_diag["flag_fuera_limite_buffer_5m"]).sum()), "Bandas reguladas fuera incluso con buffer 5 m; se conservan como diagnóstico para el mapa."),
], columns=["check", "valor", "interpretacion"])

display(bandas_quality)


,check,valor,interpretacion
0,n_bandas_raw,87615,Bandas recibidas.
1,n_bandas_candidato_clean,34450,"Bandas SER reguladas válidas por color, plazas y geometría; el límite queda como diagnóstico."
2,n_bandas_sin_color,2,Bandas sin color normalizable; se excluyen del clean.
3,n_bandas_gris,53163,Bandas grises diagnosticadas; se excluyen por no ser color SER regulado objetivo.
4,plazas_gris,331380,Plazas asociadas a bandas grises.
5,n_plazas_nulas,2,Registros con numero_plazas nulo; se excluyen del clean.
6,n_plazas_cero,0,Registros con cero plazas.
7,n_plazas_negativas,0,Registros con plazas negativas.
8,n_geometrias_invalidas,0,Geometrías nulas o inválidas.
9,n_bandas_gris_fuera_limite_estricto,53151,Bandas grises fuera del límite estricto.


**Lectura/decisión.** El clean conserva las bandas con color SER regulado, `numero_plazas` informado y geometría válida. Se excluyen las bandas sin color normalizable y los registros con `numero_plazas` nulo, porque no pueden simbolizarse ni aportar capacidad fiable en la capa cartográfica.

Las bandas grises se excluyen porque no pertenecen a los colores SER regulados objetivo del TFM (`azul`, `verde`, `alta_rotacion`, `rojo`, `naranja`). No se eliminan porque todas estén fuera del límite: una parte queda dentro o cerca del ámbito SER. La justificación correcta es semántica/cartográfica, no puramente espacial.

En cuanto al límite SER, el notebook distingue entre fuera del límite estricto y fuera con buffer de 5 m. Las bandas reguladas que siguen fuera incluso con buffer se mantienen en el clean interim porque tienen color SER válido y geometría válida. Su posible exclusión se decidirá en el notebook posterior de mapas, donde se podrá comprobar visualmente si están realmente alejadas del ámbito SER o responden a pequeñas inconsistencias geométricas.


In [12]:
ser_geoportal_bandas_aparcamiento_clean = bandas_diag.loc[bandas_keep, BANDAS_FINAL_COLUMNS].copy()
display(ser_geoportal_bandas_aparcamiento_clean.head())


,id_banda,color,numero_plazas,geometry
0,2315962,naranja,13,"LINESTRING (439261.659 4474034.444, 439260.569 4474008.677, 439261.25 4474006.359, 439263.432 4474003.905, 439266.704 4474003.087, 439295.197 4474003.224)"
1,2316848,verde,3,"LINESTRING (439894.322 4475590.35, 439878.656 4475592.016)"
2,2316855,verde,8,"LINESTRING (439941.906 4475585.266, 439900.572 4475589.766)"
3,2316980,verde,3,"LINESTRING (439668.724 4475319.033, 439661.079 4475305)"
4,2317112,azul,2,"LINESTRING (440758.049 4472149.407, 440759.416 4472159.843)"


## 8. Limpieza de `ser_calles_plazas`

**Qué mide.** `ser_calles_plazas` mide la capacidad tabular histórica de plazas SER por año, coordenadas, calle/finca, color y número de plazas.

**Uso en el TFM.** Es la fuente principal para construir capacidad SER agregable por año, barrio o calle. Se usará en fases posteriores como denominador de oferta tabular, no como geometría lineal de mapa.

**Columnas conservadas.** Se conservan `anio`, `gis_x`, `gis_y`, distrito/barrio, `calle`, `numero_finca`, `color` canónico y `numero_plazas`, porque definen localización, unidad temporal anual, tipo de plaza y capacidad.

**Columnas descartadas.** Se descartan `bateria_linea`, `esquema_documental`, flags y trazabilidad de origen. `esquema_documental` se usa solo para armonizar cambios de esquema entre años; una vez normalizadas las columnas, `anio` conserva la información temporal necesaria.

**Validaciones temporales.** Esta fuente no tiene intervalo `fecha_inicio`/`fecha_fin`: su unidad temporal es el año de publicación extraído del nombre del archivo. Por tanto, la validación temporal correcta es comprobar que el año se parsea, que pertenece a la ventana 2023–2026 y que cada archivo entra en un esquema documental esperado. No procede crear reglas de duración ni solape temporal.

**Validaciones de calidad.** Se comprueban plazas nulas, cero o negativas, color nulo, duplicados exactos, contradicciones de plazas en una misma coordenada/año y posición respecto al límite SER. Los puntos fuera del límite se diagnostican, pero no se eliminan aquí: esa decisión se revisará en notebooks posteriores de joins/mapa.


In [13]:
CALLES_FINAL_COLUMNS = [
    "anio", "gis_x", "gis_y", "cod_distrito", "distrito", "cod_barrio", "num_barrio", "barrio",
    "calle", "numero_finca", "color", "numero_plazas",
]


def diagnose_calles_plazas(items: list[dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    cleaned_parts = []
    temporal_rows = []

    for item in items:
        df = item["norm"].copy()
        anio = extract_year_from_name(item["path"])
        in_window = anio in WINDOW_CALLES

        if anio is None:
            esquema = pd.NA
            estado_temporal = "anio_no_parseable"
        elif not in_window:
            esquema = pd.NA
            estado_temporal = "anio_fuera_ventana"
        elif anio <= 2023:
            esquema = "hasta_2023"
            estado_temporal = "incluido"
        elif anio == 2024:
            esquema = "2024"
            estado_temporal = "incluido"
        else:
            esquema = "desde_2025"
            estado_temporal = "incluido"

        temporal_rows.append({
            "archivo": relpath(item["path"]),
            "anio_parseado": anio,
            "n_filas_raw": int(len(df)),
            "en_ventana_2023_2026": bool(in_window),
            "esquema_documental": esquema,
            "estado_temporal": estado_temporal,
        })

        if not in_window:
            continue

        for col in [
            "gis_x", "gis_y", "cod_distrito", "distrito", "cod_barrio", "num_barrio", "barrio",
            "calle", "numero_finca", "color", "bateria_linea", "numero_plazas",
        ]:
            if col not in df.columns:
                df[col] = pd.NA

        parsed_cod_distrito, parsed_distrito = split_code_text(df["distrito"])
        cod_distrito_raw = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
        cod_distrito = cod_distrito_raw.fillna(parsed_cod_distrito).astype("Int64")
        distrito = parsed_distrito.where(parsed_distrito.notna(), clean_text_series(df["distrito"]))

        barrio_parts = parse_barrio_text(df["barrio"])
        parsed_cod_from_barrio, parsed_barrio_simple = split_code_text(df["barrio"])
        cod_barrio_raw = pd.to_numeric(df["cod_barrio"], errors="coerce")
        num_barrio_raw = pd.to_numeric(df["num_barrio"], errors="coerce")
        num_barrio = (
            num_barrio_raw
            .fillna(barrio_parts["num_barrio_from_barrio"])
            .fillna(cod_barrio_raw.where(cod_barrio_raw <= 99))
            .fillna((cod_barrio_raw % 100).where(cod_barrio_raw > 99))
            .fillna((parsed_cod_from_barrio % 100).where(parsed_cod_from_barrio > 99, parsed_cod_from_barrio))
            .round()
            .astype("Int64")
        )
        cod_barrio = (
            cod_barrio_raw.where(cod_barrio_raw > 99)
            .fillna(parsed_cod_from_barrio.where(parsed_cod_from_barrio > 99))
            .fillna(compose_barrio_code(cod_distrito, num_barrio))
            .round()
            .astype("Int64")
        )
        barrio = (
            barrio_parts["barrio_nombre_from_barrio"]
            .where(barrio_parts["barrio_nombre_from_barrio"].notna(), parsed_barrio_simple)
            .where(lambda s: s.notna(), clean_text_series(df["barrio"]))
        )

        numero_plazas = to_numeric_series(df["numero_plazas"]).round().astype("Int64")
        out = pd.DataFrame({
            "anio": anio,
            "gis_x": to_numeric_series(df["gis_x"]),
            "gis_y": to_numeric_series(df["gis_y"]),
            "cod_distrito": cod_distrito,
            "distrito": clean_text_series(distrito),
            "cod_barrio": cod_barrio,
            "num_barrio": num_barrio,
            "barrio": clean_text_series(barrio),
            "calle": clean_text_series(df["calle"]),
            "numero_finca": clean_text_series(df["numero_finca"]),
            "color": clean_text_series(df["color"]).map(color_for_clean).astype("string"),
            "bateria_linea": clean_text_series(df["bateria_linea"]),
            "numero_plazas": numero_plazas,
            "esquema_documental": esquema,
        })
        out["flag_plazas_nulas"] = out["numero_plazas"].isna()
        out["flag_plazas_cero"] = out["numero_plazas"].fillna(-1).eq(0)
        out["flag_plazas_negativas"] = out["numero_plazas"].fillna(0).lt(0)
        out["flag_coordenadas_nulas"] = out[["gis_x", "gis_y"]].isna().any(axis=1)
        out["flag_numero_finca_sin_asignar"] = out["numero_finca"].str.upper().eq("SIN ASIGNAR").fillna(False)
        out["flag_esquema_anio"] = ~out["anio"].isin(sorted(WINDOW_CALLES))
        cleaned_parts.append(out)

    temporal_check = pd.DataFrame(temporal_rows)
    if not cleaned_parts:
        raise ValueError("No se ha limpiado ningún archivo de ser_calles_plazas en ventana 2023-2026.")
    return pd.concat(cleaned_parts, ignore_index=True), temporal_check


def duplicated_with_different_values(df: pd.DataFrame, keys: list[str], value_col: str) -> pd.DataFrame:
    return (
        df.dropna(subset=keys)
        .groupby(keys, dropna=False)[value_col]
        .nunique(dropna=True)
        .reset_index(name=f"n_{value_col}_distintos")
        .loc[lambda x: x[f"n_{value_col}_distintos"].gt(1)]
    )


ser_calles_plazas_diagnostic, calles_archivos_temporales_check = diagnose_calles_plazas(RAW_TABLES["ser_calles_plazas"])
calles_candidate_clean = ser_calles_plazas_diagnostic.loc[
    ser_calles_plazas_diagnostic["numero_plazas"].notna(),
    CALLES_FINAL_COLUMNS,
].copy()
duplicados_exactos_eliminables_calles = calles_candidate_clean.loc[calles_candidate_clean.duplicated(keep="first")].copy()
n_duplicados_exactos_detectados = int(len(duplicados_exactos_eliminables_calles))
duplicados_exactos_eliminables_por_anio = (
    duplicados_exactos_eliminables_calles
    .groupby("anio", dropna=False)
    .size()
    .astype(int)
    .to_dict()
)

calles_temporal_quality = pd.DataFrame([
    ("n_archivos_raw", int(len(calles_archivos_temporales_check)), "Archivos tabulares detectados para ser_calles_plazas."),
    ("n_archivos_incluidos_2023_2026", int(calles_archivos_temporales_check["en_ventana_2023_2026"].sum()), "Archivos incluidos por año parseado dentro de la ventana 2023-2026."),
    ("n_archivos_anio_no_parseable", int(calles_archivos_temporales_check["estado_temporal"].eq("anio_no_parseable").sum()), "Archivos cuyo año no pudo extraerse del nombre."),
    ("n_archivos_fuera_ventana", int(calles_archivos_temporales_check["estado_temporal"].eq("anio_fuera_ventana").sum()), "Archivos excluidos por estar fuera de la ventana 2023-2026."),
    ("anios_incluidos", sorted(calles_archivos_temporales_check.loc[calles_archivos_temporales_check["en_ventana_2023_2026"], "anio_parseado"].dropna().astype(int).unique().tolist()), "Años efectivamente incluidos en el diagnóstico limpio."),
], columns=["check", "valor", "interpretacion"])

dup_xy_anio = duplicated_with_different_values(
    calles_candidate_clean,
    ["anio", "gis_x", "gis_y"],
    "numero_plazas",
)
dup_xy_anio_color = duplicated_with_different_values(
    calles_candidate_clean,
    ["anio", "gis_x", "gis_y", "color"],
    "numero_plazas",
)

limite_geom_buffer_5m = limite_geom.buffer(5)
gdf_calles = gpd.GeoDataFrame(
    calles_candidate_clean.copy(),
    geometry=[
        Point(xy) if ok else None
        for xy, ok in zip(
            zip(calles_candidate_clean["gis_x"], calles_candidate_clean["gis_y"]),
            calles_candidate_clean["gis_x"].notna() & calles_candidate_clean["gis_y"].notna(),
        )
    ],
    crs="EPSG:25830",
)
calles_sin_coord = gdf_calles.geometry.isna()
calles_dentro = gdf_calles.geometry.within(limite_geom).fillna(False)
calles_dentro_buffer = gdf_calles.geometry.within(limite_geom_buffer_5m).fillna(False)
calles_limite_check = (
    gdf_calles.assign(
        _sin_coord=calles_sin_coord,
        _fuera_estricto=(~calles_sin_coord & ~calles_dentro),
        _fuera_buffer_5m=(~calles_sin_coord & ~calles_dentro_buffer),
    )
    .groupby("anio", dropna=False)
    .agg(
        n_registros=("anio", "size"),
        n_fuera_limite_estricto=("_fuera_estricto", "sum"),
        n_fuera_limite_buffer_5m=("_fuera_buffer_5m", "sum"),
    )
    .reset_index()
)
calles_limite_check["pct_fuera_limite_estricto"] = (
    calles_limite_check["n_fuera_limite_estricto"] / calles_limite_check["n_registros"] * 100
).round(3)
calles_limite_check["pct_fuera_limite_buffer_5m"] = (
    calles_limite_check["n_fuera_limite_buffer_5m"] / calles_limite_check["n_registros"] * 100
).round(3)

calles_quality = pd.DataFrame([
    ("n_filas_diagnostico", int(len(ser_calles_plazas_diagnostic)), "Registros antes de filtrar plazas nulas."),
    ("n_filas_candidato_clean", int(len(calles_candidate_clean)), "Registros con plazas informadas antes de eliminar duplicados exactos."),
    ("n_plazas_nulas_excluibles", int(ser_calles_plazas_diagnostic["flag_plazas_nulas"].sum()), "Registros sin numero_plazas; no sirven como capacidad."),
    ("n_plazas_cero", int(ser_calles_plazas_diagnostic["flag_plazas_cero"].sum()), "Registros con cero plazas."),
    ("n_plazas_negativas", int(ser_calles_plazas_diagnostic["flag_plazas_negativas"].sum()), "Registros con plazas negativas."),
    ("n_color_nulo", int(calles_candidate_clean["color"].isna().sum()), "Registros candidato clean sin color normalizado."),
    ("n_duplicados_exactos_detectados", n_duplicados_exactos_detectados, "Duplicados exactos en columnas finales; se eliminarán del clean final."),
    ("duplicados_exactos_eliminables_por_anio", duplicados_exactos_eliminables_por_anio, "Año de los registros duplicados exactos que se eliminan; la clave incluye anio."),
    ("n_misma_xy_anio_con_plazas_distintas", int(len(dup_xy_anio)), "Misma coordenada y año con valores distintos de numero_plazas."),
    ("n_misma_xy_anio_color_con_plazas_distintas", int(len(dup_xy_anio_color)), "Misma coordenada, año y color con valores distintos de numero_plazas."),
    ("n_fuera_limite_estricto_total", int(calles_limite_check["n_fuera_limite_estricto"].sum()), "Puntos fuera del límite SER estricto; diagnóstico, no filtrado."),
    ("n_fuera_limite_buffer_5m_total", int(calles_limite_check["n_fuera_limite_buffer_5m"].sum()), "Puntos fuera del límite SER con tolerancia 5 m; diagnóstico, no filtrado."),
], columns=["check", "valor", "interpretacion"])

print("A. Validación temporal de archivos ser_calles_plazas")
display(calles_temporal_quality)
display(calles_archivos_temporales_check)
print("B. Validaciones de calidad de registros ser_calles_plazas")
display(calles_quality)
display(calles_limite_check)


A. Validación temporal de archivos ser_calles_plazas


,check,valor,interpretacion
0,n_archivos_raw,4,Archivos tabulares detectados para ser_calles_plazas.
1,n_archivos_incluidos_2023_2026,4,Archivos incluidos por año parseado dentro de la ventana 2023-2026.
2,n_archivos_anio_no_parseable,0,Archivos cuyo año no pudo extraerse del nombre.
3,n_archivos_fuera_ventana,0,Archivos excluidos por estar fuera de la ventana 2023-2026.
4,anios_incluidos,"[2023, 2024, 2025, 2026]",Años efectivamente incluidos en el diagnóstico limpio.


,archivo,anio_parseado,n_filas_raw,en_ventana_2023_2026,esquema_documental,estado_temporal
0,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2023.csv,2023,32111,True,hasta_2023,incluido
1,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2024.csv,2024,33700,True,2024,incluido
2,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2025.csv,2025,34583,True,desde_2025,incluido
3,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2026.csv,2026,34520,True,desde_2025,incluido


B. Validaciones de calidad de registros ser_calles_plazas


,check,valor,interpretacion
0,n_filas_diagnostico,134914,Registros antes de filtrar plazas nulas.
1,n_filas_candidato_clean,134913,Registros con plazas informadas antes de eliminar duplicados exactos.
2,n_plazas_nulas_excluibles,1,Registros sin numero_plazas; no sirven como capacidad.
3,n_plazas_cero,0,Registros con cero plazas.
4,n_plazas_negativas,0,Registros con plazas negativas.
5,n_color_nulo,0,Registros candidato clean sin color normalizado.
6,n_duplicados_exactos_detectados,4,Duplicados exactos en columnas finales; se eliminarán del clean final.
7,duplicados_exactos_eliminables_por_anio,"{2023: 1, 2024: 1, 2025: 1, 2026: 1}",Año de los registros duplicados exactos que se eliminan; la clave incluye anio.
8,n_misma_xy_anio_con_plazas_distintas,0,Misma coordenada y año con valores distintos de numero_plazas.
9,n_misma_xy_anio_color_con_plazas_distintas,0,"Misma coordenada, año y color con valores distintos de numero_plazas."


,anio,n_registros,n_fuera_limite_estricto,n_fuera_limite_buffer_5m,pct_fuera_limite_estricto,pct_fuera_limite_buffer_5m
0,2023,32110,32,5,0.100,0.016
1,2024,33700,35,6,0.104,0.018
2,2025,34583,38,6,0.110,0.017
3,2026,34520,33,4,0.096,0.012


**Lectura/decisión.** La validación temporal confirma que los cuatro archivos anuales de `ser_calles_plazas` se parsean correctamente desde el nombre del fichero y corresponden a 2023, 2024, 2025 y 2026. No hay archivos con año no parseable ni archivos fuera de la ventana temporal del TFM, por lo que no se excluye información anual de forma silenciosa.

Se excluye cualquier registro con `numero_plazas` nulo, porque no puede utilizarse como capacidad. No hay plazas cero, plazas negativas ni colores nulos.

Los duplicados exactos se calculan sobre las columnas finales, incluyendo `anio`. Por tanto, no se elimina una misma calle por aparecer en años distintos: solo se elimina una repetición idéntica dentro del mismo año, con las mismas coordenadas, barrio, calle, finca, color y plazas. El check `duplicados_exactos_eliminables_por_anio` permite verificar en qué año o años se localizan esos registros eliminables.

No aparecen contradicciones de `numero_plazas` para una misma coordenada y año, ni para una misma coordenada, año y color. Esto es más relevante que contar repeticiones por calle, porque una calle puede tener múltiples fincas o segmentos legítimos.

La validación espacial detecta puntos fuera del límite SER estricto y fuera incluso con buffer de 5 m. No se eliminan en este notebook, porque `ser_calles_plazas` será evaluada de nuevo al construir joins y mapas. Por ahora quedan como incidencia espacial trazada, no como criterio de exclusión.


In [19]:
ser_calles_plazas_clean = calles_candidate_clean.drop_duplicates().reset_index(drop=True)
display(ser_calles_plazas_clean.head())


,anio,gis_x,gis_y,cod_distrito,distrito,cod_barrio,num_barrio,barrio,calle,numero_finca,color,numero_plazas
0,2023,439592.91,4473566.23,1,CENTRO,101,1,PALACIO,"AGUAS, CALLE, DE LAS",2,verde,7
1,2023,439569.07,4473598.77,1,CENTRO,101,1,PALACIO,"AGUAS, CALLE, DE LAS",8,verde,5
2,2023,439578.18,4473498.76,1,CENTRO,101,1,PALACIO,"AGUILA, CALLE, DEL",3,verde,1
3,2023,439574.49,4473493.40,1,CENTRO,101,1,PALACIO,"AGUILA, CALLE, DEL",5,verde,1
4,2023,439559.19,4473471.82,1,CENTRO,101,1,PALACIO,"AGUILA, CALLE, DEL",12,verde,9


## 9. Limpieza de `ser_parquimetros`

**Qué mide.** `ser_parquimetros` describe la infraestructura SER de parquímetros, con matrícula, vigencia temporal, calle/finca y coordenadas.

**Uso en el TFM.** Será clave para enlazar tiques con infraestructura mediante `matricula`, aunque este notebook no construye todavía ese join.

**Columnas conservadas.** Se conservan coordenadas (`gis_x`, `gis_y`, `longitud`, `latitud`), fechas de alta/baja, distrito/barrio, `calle`, `numero_finca` y `matricula`, porque definen ubicación, vigencia e identificador operativo.

**Columnas descartadas.** Se descartan flags temporales y trazabilidad de origen. Los flags se usan solo para diagnosticar y decidir si hay registros sin matrícula, fechas inválidas, bajas fuera de ventana o problemas de vigencia.

**Validaciones temporales.** Se comprueba que las fechas de alta/baja sean parseables, que `fecha_de_baja` nula se trate como parquímetro activo, que no haya vigencias invertidas (`fecha_de_alta > fecha_de_baja`) y que la vigencia interseque la ventana operativa 2023–2026. No se impone una duración máxima: en infraestructura urbana una vigencia larga puede ser perfectamente válida y un umbral arbitrario generaría ruido.

**Validaciones de calidad.** Se comprueban duplicados exactos, registros sin matrícula, coordenadas nulas, reutilización de matrícula, intervalos de vigencia solapados y posición respecto al límite SER. La eliminación operativa se limita a registros que no pueden participar en el pipeline 2023–2026.


In [15]:
PARQUIMETROS_FINAL_COLUMNS = [
    "gis_x", "gis_y", "fecha_de_alta", "fecha_de_baja", "cod_distrito", "distrito",
    "cod_barrio", "num_barrio", "barrio", "calle", "numero_finca", "matricula", "longitud", "latitud",
]
FUTURE_DATE = pd.Timestamp("2099-12-31")


def diagnose_ser_parquimetros(items: list[dict[str, Any]]) -> tuple[pd.DataFrame, int, dict[str, Any]]:
    if len(items) != 1:
        print(f"Aviso: ser_parquimetros tiene {len(items)} archivos tabulares; se usa el primero.")
    df = items[0]["norm"].copy()
    for col in PARQUIMETROS_FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    cod_distrito = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
    cod_barrio_raw = pd.to_numeric(df["cod_barrio"], errors="coerce")
    num_barrio_raw = pd.to_numeric(df["num_barrio"], errors="coerce")
    num_barrio = (
        num_barrio_raw
        .fillna(cod_barrio_raw.where(cod_barrio_raw <= 99))
        .fillna((cod_barrio_raw % 100).where(cod_barrio_raw > 99))
        .round()
        .astype("Int64")
    )
    cod_barrio = (
        cod_barrio_raw.where(cod_barrio_raw > 99)
        .fillna(compose_barrio_code(cod_distrito, num_barrio))
        .round()
        .astype("Int64")
    )
    fecha_alta_raw = clean_text_series(df["fecha_de_alta"])
    fecha_baja_raw = clean_text_series(df["fecha_de_baja"])

    diagnostic = pd.DataFrame({
        "gis_x": to_numeric_series(df["gis_x"]),
        "gis_y": to_numeric_series(df["gis_y"]),
        "fecha_de_alta": pd.to_datetime(fecha_alta_raw, errors="coerce"),
        "fecha_de_baja": pd.to_datetime(fecha_baja_raw, errors="coerce"),
        "cod_distrito": cod_distrito,
        "distrito": clean_text_series(df["distrito"]),
        "cod_barrio": cod_barrio,
        "num_barrio": num_barrio,
        "barrio": clean_text_series(df["barrio"]),
        "calle": clean_text_series(df["calle"]),
        "numero_finca": clean_text_series(df["numero_finca"]),
        "matricula": clean_identifier_series(df["matricula"]),
        "longitud": to_numeric_series(df["longitud"]),
        "latitud": to_numeric_series(df["latitud"]),
    })
    diagnostic["flag_calle_sin_asignar"] = diagnostic["calle"].str.upper().eq("SIN ASIGNAR").fillna(False)
    diagnostic["flag_parquimetro_sin_coordenadas"] = diagnostic[["gis_x", "gis_y"]].isna().any(axis=1)
    diagnostic["flag_fecha_alta_invalida"] = fecha_alta_raw.notna() & diagnostic["fecha_de_alta"].isna()
    diagnostic["flag_fecha_baja_invalida"] = fecha_baja_raw.notna() & diagnostic["fecha_de_baja"].isna()
    diagnostic["flag_fecha_alta_nula"] = diagnostic["fecha_de_alta"].isna()
    diagnostic["flag_fecha_baja_nula"] = diagnostic["fecha_de_baja"].isna()
    diagnostic["flag_parquimetro_dado_baja"] = diagnostic["fecha_de_baja"].notna()
    diagnostic["flag_vigencia_invertida"] = (
        diagnostic["fecha_de_alta"].notna()
        & diagnostic["fecha_de_baja"].notna()
        & diagnostic["fecha_de_alta"].gt(diagnostic["fecha_de_baja"])
    )
    diagnostic["flag_vigencia_fuera_ventana_previa"] = (
        diagnostic["fecha_de_baja"].notna()
        & diagnostic["fecha_de_baja"].lt(SER_WINDOW_START)
    )
    diagnostic["flag_vigencia_fuera_ventana_posterior"] = (
        diagnostic["fecha_de_alta"].notna()
        & diagnostic["fecha_de_alta"].gt(SER_WINDOW_END)
    )
    diagnostic["flag_vigencia_interseca_ventana"] = (
        ~diagnostic["flag_vigencia_fuera_ventana_previa"]
        & ~diagnostic["flag_vigencia_fuera_ventana_posterior"]
    )

    # Duración solo para vigencias cerradas. No se usa como filtro porque una vida útil larga de un parquímetro no es incoherente por sí misma.
    diagnostic["duracion_vigencia_cerrada_dias"] = (
        diagnostic["fecha_de_baja"] - diagnostic["fecha_de_alta"]
    ).dt.days

    duplicados_exactos_eliminables = diagnostic.loc[
        diagnostic.duplicated(subset=PARQUIMETROS_FINAL_COLUMNS, keep="first"),
        PARQUIMETROS_FINAL_COLUMNS,
    ].copy()
    n_before = len(diagnostic)
    diagnostic = diagnostic.drop_duplicates(subset=PARQUIMETROS_FINAL_COLUMNS).reset_index(drop=True)
    n_exact_removed = n_before - len(diagnostic)
    exact_duplicate_summary = {
        "n_registros_eliminables": int(len(duplicados_exactos_eliminables)),
        "matriculas_afectadas": sorted(duplicados_exactos_eliminables["matricula"].dropna().astype(str).unique().tolist()),
        "n_sin_matricula": int(duplicados_exactos_eliminables["matricula"].isna().sum()),
    }
    return diagnostic, n_exact_removed, exact_duplicate_summary


def matricula_interval_overlaps(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    interval_df = df.loc[df["matricula"].notna()].copy()
    interval_df["inicio"] = interval_df["fecha_de_alta"].fillna(SER_WINDOW_START)
    interval_df["fin"] = interval_df["fecha_de_baja"].fillna(FUTURE_DATE)
    for matricula, group in interval_df.sort_values(["matricula", "inicio", "fin"]).groupby("matricula"):
        if len(group) < 2:
            continue
        records = group.reset_index(drop=True)
        for idx in range(len(records) - 1):
            current = records.iloc[idx]
            nxt = records.iloc[idx + 1]
            # Intervalos semiabiertos: si una baja y otra alta coinciden el mismo día, no se cuenta como solape real.
            if current["fin"] > nxt["inicio"]:
                rows.append({
                    "matricula": matricula,
                    "inicio_a": current["inicio"],
                    "fin_a": current["fin"],
                    "inicio_b": nxt["inicio"],
                    "fin_b": nxt["fin"],
                    "gis_x_a": current["gis_x"],
                    "gis_y_a": current["gis_y"],
                    "gis_x_b": nxt["gis_x"],
                    "gis_y_b": nxt["gis_y"],
                })
    return pd.DataFrame(rows)


ser_parquimetros_diagnostic, parquimetros_exact_removed, parquimetros_exact_duplicate_summary = diagnose_ser_parquimetros(RAW_TABLES["ser_parquimetros"])

parquimetros_keep = (
    ser_parquimetros_diagnostic["matricula"].notna()
    & ~ser_parquimetros_diagnostic["flag_fecha_alta_invalida"]
    & ~ser_parquimetros_diagnostic["flag_fecha_baja_invalida"]
    & ~ser_parquimetros_diagnostic["flag_vigencia_invertida"]
    & ser_parquimetros_diagnostic["flag_vigencia_interseca_ventana"]
)
parquimetros_candidate_clean = ser_parquimetros_diagnostic.loc[parquimetros_keep, PARQUIMETROS_FINAL_COLUMNS].copy()

# La reutilización de matrícula relevante para el pipeline se diagnostica sobre el candidato clean,
# no sobre registros históricos excluidos por baja anterior a 2023.
mat_groups = parquimetros_candidate_clean.loc[parquimetros_candidate_clean["matricula"].notna()].groupby("matricula", dropna=False)
duplicados_matricula_resumen = mat_groups.agg(
    n_registros=("matricula", "size"),
    n_coordenadas=("gis_x", lambda s: parquimetros_candidate_clean.loc[s.index, ["gis_x", "gis_y"]].drop_duplicates().shape[0]),
    n_fechas=("fecha_de_alta", lambda s: parquimetros_candidate_clean.loc[s.index, ["fecha_de_alta", "fecha_de_baja"]].drop_duplicates().shape[0]),
).reset_index()
duplicados_matricula_resumen = duplicados_matricula_resumen.loc[duplicados_matricula_resumen["n_registros"].gt(1)]

# Los solapes temporales también se evalúan sobre el candidato clean, ya filtrado por ventana operativa.
matricula_solapes = matricula_interval_overlaps(parquimetros_candidate_clean)
if not matricula_solapes.empty:
    matricula_solapes = matricula_solapes.assign(
        misma_coord=lambda df: (
            df["gis_x_a"].round(3).eq(df["gis_x_b"].round(3))
            & df["gis_y_a"].round(3).eq(df["gis_y_b"].round(3))
        )
    )
n_solapes_misma_coord = int(matricula_solapes["misma_coord"].sum()) if not matricula_solapes.empty else 0
n_solapes_coord_distinta = int((~matricula_solapes["misma_coord"]).sum()) if not matricula_solapes.empty else 0

gdf_parquimetros = gpd.GeoDataFrame(
    parquimetros_candidate_clean.copy(),
    geometry=[
        Point(xy) if ok else None
        for xy, ok in zip(
            zip(parquimetros_candidate_clean["gis_x"], parquimetros_candidate_clean["gis_y"]),
            parquimetros_candidate_clean["gis_x"].notna() & parquimetros_candidate_clean["gis_y"].notna(),
        )
    ],
    crs="EPSG:25830",
)
parq_sin_coord = gdf_parquimetros.geometry.isna()
parq_dentro = gdf_parquimetros.geometry.within(limite_geom).fillna(False)
parq_dentro_buffer = gdf_parquimetros.geometry.within(limite_geom.buffer(5)).fillna(False)

parquimetros_quality = pd.DataFrame([
    ("n_filas_diagnostico", int(len(ser_parquimetros_diagnostic)), "Registros tras eliminar duplicados exactos evidentes."),
    ("n_filas_candidato_clean", int(len(parquimetros_candidate_clean)), "Registros con matrícula válida y vigencia que interseca la ventana 2023-2026."),
    ("n_duplicados_exactos_eliminados", int(parquimetros_exact_removed), "Duplicados exactos eliminados antes del clean."),
    ("duplicados_exactos_matriculas_afectadas", parquimetros_exact_duplicate_summary["matriculas_afectadas"], "Matrículas afectadas por duplicados exactos eliminados."),
    ("n_sin_matricula_excluibles", int(ser_parquimetros_diagnostic["matricula"].isna().sum()), "Registros sin matrícula; no pueden enlazar con tiques."),
    ("n_baja_antes_2023_excluibles", int(ser_parquimetros_diagnostic["flag_vigencia_fuera_ventana_previa"].sum()), "Parquímetros dados de baja antes de la ventana 2023-2026."),
    ("n_alta_despues_2026_excluibles", int(ser_parquimetros_diagnostic["flag_vigencia_fuera_ventana_posterior"].sum()), "Parquímetros cuya alta empieza después de la ventana 2023-2026."),
    ("n_fecha_alta_invalida", int(ser_parquimetros_diagnostic["flag_fecha_alta_invalida"].sum()), "Textos de fecha_alta no parseables."),
    ("n_fecha_baja_invalida", int(ser_parquimetros_diagnostic["flag_fecha_baja_invalida"].sum()), "Textos de fecha_baja no parseables."),
    ("n_fecha_alta_nula", int(ser_parquimetros_diagnostic["flag_fecha_alta_nula"].sum()), "Registros sin fecha_de_alta; si existieran, se interpretarían como alta previa/desconocida solo para diagnóstico de intervalos."),
    ("n_fecha_baja_nula_activos", int(ser_parquimetros_diagnostic["flag_fecha_baja_nula"].sum()), "Registros sin fecha_de_baja; se interpretan como parquímetros activos."),
    ("n_vigencias_invertidas_excluibles", int(ser_parquimetros_diagnostic["flag_vigencia_invertida"].sum()), "Registros con fecha_de_alta posterior a fecha_de_baja."),
    ("n_sin_coordenadas", int(ser_parquimetros_diagnostic["flag_parquimetro_sin_coordenadas"].sum()), "Registros sin coordenadas."),
    ("n_matriculas_reutilizadas", int(len(duplicados_matricula_resumen)), "Matrículas con más de un registro dentro del candidato clean."),
    ("n_matriculas_con_fechas_distintas", int((duplicados_matricula_resumen["n_fechas"].gt(1)).sum()), "Matrículas repetidas con fechas de vigencia distintas dentro del candidato clean."),
    ("n_matriculas_con_coordenadas_distintas", int((duplicados_matricula_resumen["n_coordenadas"].gt(1)).sum()), "Matrículas repetidas con coordenadas distintas dentro del candidato clean."),
    ("n_matriculas_con_intervalos_solapados", int(matricula_solapes["matricula"].nunique()) if not matricula_solapes.empty else 0, "Matrículas con intervalos de vigencia solapados usando criterio estricto dentro del candidato clean."),
    ("n_solapes_misma_coord", n_solapes_misma_coord, "Solapes temporales cuya coordenada coincide redondeando a 1 mm."),
    ("n_solapes_coord_distinta", n_solapes_coord_distinta, "Solapes temporales con coordenadas distintas."),
    ("n_fuera_limite_estricto", int((~parq_sin_coord & ~parq_dentro).sum()), "Parquímetros candidato clean fuera del límite SER estricto."),
    ("n_fuera_limite_buffer_5m", int((~parq_sin_coord & ~parq_dentro_buffer).sum()), "Parquímetros candidato clean fuera del límite SER con tolerancia 5 m."),
], columns=["check", "valor", "interpretacion"])

display(parquimetros_quality)

if not matricula_solapes.empty:
    solapes_cols = [
        "matricula",
        "inicio_a", "fin_a",
        "inicio_b", "fin_b",
        "gis_x_a", "gis_y_a",
        "gis_x_b", "gis_y_b",
        "misma_coord",
    ]
    display(matricula_solapes[solapes_cols].head(5))


,check,valor,interpretacion
0,n_filas_diagnostico,6245,Registros tras eliminar duplicados exactos evidentes.
1,n_filas_candidato_clean,4772,Registros con matrícula válida y vigencia que interseca la ventana 2023-2026.
2,n_duplicados_exactos_eliminados,1,Duplicados exactos eliminados antes del clean.
3,duplicados_exactos_matriculas_afectadas,[],Matrículas afectadas por duplicados exactos eliminados.
4,n_sin_matricula_excluibles,27,Registros sin matrícula; no pueden enlazar con tiques.
5,n_baja_antes_2023_excluibles,1450,Parquímetros dados de baja antes de la ventana 2023-2026.
6,n_alta_despues_2026_excluibles,0,Parquímetros cuya alta empieza después de la ventana 2023-2026.
7,n_fecha_alta_invalida,0,Textos de fecha_alta no parseables.
8,n_fecha_baja_invalida,0,Textos de fecha_baja no parseables.
9,n_fecha_alta_nula,0,"Registros sin fecha_de_alta; si existieran, se interpretarían como alta previa/desconocida solo para diagnóstico de intervalos."


**Lectura/decisión.** La validación temporal de `ser_parquimetros` confirma que las fechas de alta y baja son parseables, que no existen vigencias invertidas y que no hay registros cuya alta comience después de 2026. Las bajas nulas se interpretan como parquímetros activos, lo que afecta a 4679 registros. El filtro conserva 4772 parquímetros con matrícula válida y vigencia que intersecta la ventana 2023–2026; se excluyen 27 registros sin matrícula, 1450 dados de baja antes de 2023 y se elimina el duplicado exacto.

No se aplica una regla de duración máxima. En esta fuente, una vigencia larga no es necesariamente incoherente: un parquímetro puede permanecer activo muchos años. Por eso la duración cerrada queda solo como variable de diagnóstico interno y no como criterio de filtrado.

Los duplicados exactos se eliminan solo si coinciden todas las columnas finales: coordenadas, fechas, distrito/barrio, calle/finca, matrícula, longitud y latitud. Por tanto, no se eliminan reutilizaciones históricas de matrícula con fechas o coordenadas distintas.

Las reutilizaciones de matrícula se diagnostican sobre el candidato clean, ya filtrado por matrícula y vigencia compatible con 2023–2026. Si aparece alguna matrícula con intervalo de vigencia solapado usando criterio estricto, queda trazada para el notebook de joins con tiques, donde se decidirá usando la fecha real de cada tique.

Respecto al límite SER, los parquímetros fuera del límite estricto o incluso fuera con buffer de 5 m no se eliminan en este notebook, porque `ser_parquimetros` será evaluada de nuevo al construir joins y mapas. Por ahora quedan como incidencia espacial trazada, no como criterio de exclusión.


In [16]:
ser_parquimetros_clean = parquimetros_candidate_clean.copy()
display(ser_parquimetros_clean.head())


,gis_x,gis_y,fecha_de_alta,fecha_de_baja,cod_distrito,distrito,cod_barrio,num_barrio,barrio,calle,numero_finca,matricula,longitud,latitud
0,439608.190788,4.473550e+06,2014-05-28,NaT,1,CENTRO,101,1,PALACIO,"AGUAS, CALLE, DE LAS",1,301110056,-3.711774,40.410379
1,439757.570954,4.473795e+06,2014-05-27,NaT,1,CENTRO,101,1,PALACIO,"ALMENDRO, CALLE, DEL",14,301110042,-3.710037,40.412592
2,439724.473069,4.474317e+06,2014-05-26,NaT,1,CENTRO,101,1,PALACIO,"AMNISTIA, CALLE, DE LA",6,301110014,-3.710476,40.417300
3,439923.681109,4.474529e+06,2014-05-26,NaT,1,CENTRO,101,1,PALACIO,"ANGELES, COSTANILLA, DE LOS",11,301110012,-3.708149,40.419220
4,439735.231748,4.474490e+06,2014-05-26,NaT,1,CENTRO,101,1,PALACIO,"ARRIETA, CALLE, DE",8,301110010,-3.710366,40.418855


## 10. Validaciones cruzadas ligeras entre fuentes limpias

Esta sección compara fuentes ya limpias sin construir joins finales. El objetivo es detectar incoherencias tempranas entre la capa cartográfica que se usará para mapas y las fuentes tabulares que se usarán para capacidad o enlace con tiques.

Se mantienen dos validaciones ligeras: comparación de plazas por color entre bandas Geoportal y `ser_calles_plazas` 2025, y cobertura aproximada de nombres de calle de parquímetros dentro de `ser_calles_plazas`. Estas validaciones no eliminan registros ni corrigen geometrías.


In [17]:
def points_from_xy(df: pd.DataFrame, x_col: str = "gis_x", y_col: str = "gis_y") -> gpd.GeoDataFrame:
    valid = df[x_col].notna() & df[y_col].notna()
    points = [Point(xy) if ok else None for xy, ok in zip(zip(df[x_col], df[y_col]), valid)]
    return gpd.GeoDataFrame(df.copy(), geometry=points, crs="EPSG:25830")


geo_color = (
    ser_geoportal_bandas_aparcamiento_clean
    .assign(color_norm=lambda df: df["color"].str.replace("_", " ", regex=False))
    .groupby("color_norm", dropna=False)["numero_plazas"]
    .sum(min_count=1)
    .reset_index(name="plazas_geoportal")
)
calles_2025_color = (
    ser_calles_plazas_clean
    .loc[lambda df: df["anio"].eq(2025)]
    .assign(color_norm=lambda df: df["color"].str.replace("_", " ", regex=False))
    .groupby("color_norm", dropna=False)["numero_plazas"]
    .sum(min_count=1)
    .reset_index(name="plazas_calles_2025")
)
comparacion_plazas_color = geo_color.merge(calles_2025_color, on="color_norm", how="outer")
comparacion_plazas_color[["plazas_geoportal", "plazas_calles_2025"]] = comparacion_plazas_color[["plazas_geoportal", "plazas_calles_2025"]].fillna(0)
comparacion_plazas_color["diferencia_abs"] = comparacion_plazas_color["plazas_geoportal"] - comparacion_plazas_color["plazas_calles_2025"]
comparacion_plazas_color["diferencia_pct"] = np.where(
    comparacion_plazas_color["plazas_calles_2025"].ne(0),
    comparacion_plazas_color["diferencia_abs"] / comparacion_plazas_color["plazas_calles_2025"] * 100,
    np.nan,
)
comparacion_plazas_color["diferencia_pct"] = comparacion_plazas_color["diferencia_pct"].round(2)
comparacion_plazas_color = comparacion_plazas_color.sort_values("color_norm", na_position="last")

calles_parq_norm = set(ser_parquimetros_clean["calle"].map(normalize_street_name).dropna())
calles_cap_norm = set(ser_calles_plazas_clean["calle"].map(normalize_street_name).dropna())
calles_parquimetros_no_en_calles = sorted(calles_parq_norm - calles_cap_norm)
calles_nombre_check = pd.DataFrame([{
    "n_calles_parquimetros": int(len(calles_parq_norm)),
    "n_calles_parquimetros_en_calles_plazas": int(len(calles_parq_norm & calles_cap_norm)),
    "pct_calles_parquimetros_en_calles_plazas": round(len(calles_parq_norm & calles_cap_norm) / len(calles_parq_norm) * 100, 3) if calles_parq_norm else np.nan,
    "ejemplos_no_encontrados": calles_parquimetros_no_en_calles[:20],
}])

print("D. Comparación de plazas por color: Geoportal reguladas vs calles/plazas 2025")
display(comparacion_plazas_color)
print("E. Validación ligera de nombres de calle parquímetros vs calles/plazas")
display(calles_nombre_check)


D. Comparación de plazas por color: Geoportal reguladas vs calles/plazas 2025


,color_norm,plazas_geoportal,plazas_calles_2025,diferencia_abs,diferencia_pct
0,alta rotacion,372,372,0,0.00
1,azul,21183,21307,-124,-0.58
2,naranja,1464,1464,0,0.00
3,rojo,342,358,-16,-4.47
4,verde,157644,157888,-244,-0.15


E. Validación ligera de nombres de calle parquímetros vs calles/plazas


,n_calles_parquimetros,n_calles_parquimetros_en_calles_plazas,pct_calles_parquimetros_en_calles_plazas,ejemplos_no_encontrados
0,1791,1756,98.046,"[alabastro del, bejar de, benito valderas de, caoba de la, caolin del, carmen del, cenicero de, circon del, circonita de la, cordon del, cuarzo del, el esco..."


**Lectura/decisión.** La comparación por color muestra coherencia alta entre la capa lineal Geoportal y `ser_calles_plazas` 2025. Alta rotación y naranja coinciden exactamente; azul difiere en -124 plazas (-0,58 %), verde en -244 plazas (-0,15 %) y rojo en -16 plazas (-4,47 %). En términos absolutos y relativos, la discrepancia global es pequeña y compatible con diferencias de actualización, geometría o codificación entre fuentes.

La validación de nombres de calle muestra que el 98,046 % de las calles normalizadas de parquímetros aparece también en `ser_calles_plazas`. Esta cobertura es suficiente para continuar con futuras pruebas de join por calle, aunque los ejemplos no encontrados deberán revisarse cuando se construya el notebook específico de joins.

La decisión es mantener ambas validaciones como evidencia de compatibilidad inicial, sin crear todavía joins finales ni eliminar registros.


## 11. Escritura de salidas limpias

Se escriben exactamente seis salidas limpias individuales en `data/interim/ser/...`. No se escribe ninguna salida en `data/processed`, ni agregados, ni paneles, ni métricas proxy.


In [18]:
ensure_parquet_engine()

clean_outputs = {
    "ser_zonas": ser_zonas_clean,
    "ser_geoportal_limite_ser": ser_geoportal_limite_ser_clean,
    "ser_geoportal_barrios_ser": ser_geoportal_barrios_ser_clean,
    "ser_geoportal_bandas_aparcamiento": ser_geoportal_bandas_aparcamiento_clean,
    "ser_calles_plazas": ser_calles_plazas_clean,
    "ser_parquimetros": ser_parquimetros_clean,
}

expected_output_names = {
    "ser_zonas_clean.parquet",
    "ser_geoportal_limite_ser_clean.parquet",
    "ser_geoportal_barrios_ser_clean.parquet",
    "ser_geoportal_bandas_aparcamiento_clean.parquet",
    "ser_calles_plazas_clean.parquet",
    "ser_parquimetros_clean.parquet",
}
actual_output_names = {Path(catalog_ser.loc[catalog_ser["dataset_id"].eq(ds), "archivo_interim"].iloc[0]).name for ds in clean_outputs}
if actual_output_names != expected_output_names:
    raise ValueError(f"Las salidas catalogadas no coinciden exactamente: {actual_output_names}")

write_rows = []
for dataset_id in TARGET_DATASET_IDS:
    df = clean_outputs[dataset_id]
    if df.empty:
        raise ValueError(f"La salida limpia de {dataset_id} esta vacia; no se escribe.")
    out_rel = catalog_ser.loc[catalog_ser["dataset_id"].eq(dataset_id), "archivo_interim"].iloc[0]
    out_path = ROOT / out_rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(df, gpd.GeoDataFrame):
        df.to_parquet(out_path, index=False)
    else:
        df.to_parquet(out_path, index=False)
    write_rows.append({
        "dataset_id": dataset_id,
        "archivo_interim": relpath(out_path),
        "shape_escrita": df.shape,
        "size_mb": round(out_path.stat().st_size / 1024**2, 3),
        "estado_escritura": "OK",
    })

write_check = pd.DataFrame(write_rows)
display(write_check)


,dataset_id,archivo_interim,shape_escrita,size_mb,estado_escritura
0,ser_zonas,data/interim/ser/ser_zonas/ser_zonas_clean.parquet,"(6720, 9)",0.082,OK
1,ser_geoportal_limite_ser,data/interim/ser/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet,"(1, 3)",0.043,OK
2,ser_geoportal_barrios_ser,data/interim/ser/ser_geoportal_barrios_ser/ser_geoportal_barrios_ser_clean.parquet,"(66, 7)",0.081,OK
3,ser_geoportal_bandas_aparcamiento,data/interim/ser/ser_geoportal_bandas_aparcamiento/ser_geoportal_bandas_aparcamiento_clean.parquet,"(34450, 4)",1.273,OK
4,ser_calles_plazas,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,"(134909, 12)",1.360,OK
5,ser_parquimetros,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,"(4772, 14)",0.239,OK


**Lectura/decisión.** La escritura genera exactamente seis Parquet limpios individuales, todos con estado `OK`: `ser_zonas`, límite SER, barrios SER, bandas de aparcamiento, calles/plazas y parquímetros.

Los shapes escritos reflejan los filtrados aplicados en esta limpieza: barrios SER reales, bandas reguladas válidas por color/plazas/geometría, registros de calles/plazas con capacidad informada y sin duplicados exactos, y parquímetros operativos para la ventana del TFM. No aparecen salidas agregadas ni processed, por lo que el notebook se mantiene dentro del alcance de limpieza individual.
